### Initialisation

In [10]:
import pandas as pd
import duckdb
from pathlib import Path

### Helper Functions

In [11]:
# Helpers for safely quoting column names in SQL
def quote_ident(name):
    return '"' + name.replace('"', '""') + '"'

### Exploratory Data Analysis

In [12]:
CSV_FILE = Path.cwd() / "usdot_insurance_events_2026-10-12.csv"

if not CSV_FILE.exists():
    raise FileNotFoundError(f"CSV not found: {CSV_FILE}")

print(f"Dataset: {CSV_FILE}")
print(f"File size: {CSV_FILE.stat().st_size / 1024**3:.2f} GB")

# Use an in-memory DuckDB database. The source CSV stays untouched.
con = duckdb.connect(":memory:")
con.execute("SET memory_limit = '6GB'")
con.execute("SET threads = 4")

# Use a temporary directory only if DuckDB needs to spill data.
# This does not create a copy of the source CSV.
temp_dir = Path.cwd() / "duckdb_temp"
temp_dir.mkdir(exist_ok=True)
con.execute(f"SET temp_directory = '{temp_dir.as_posix()}'")

csv_path = CSV_FILE.resolve().as_posix().replace("'", "''")

con.execute(f"""
    CREATE VIEW raw AS
    SELECT *
    FROM read_csv_auto(
        '{csv_path}',
        sample_size = 100000
    )
""")

Dataset: c:\Users\hi\Documents\ROSELLA ASSIGNMENT\usdot_insurance_events_2026-10-12.csv
File size: 43.87 GB


In [4]:
# --------------------------------------------------
# 1. Schema: column names and inferred data types
# --------------------------------------------------

print("\n" + "=" * 80)
print("1. SCHEMA")
print("=" * 80)

schema = con.execute("DESCRIBE raw").df()
print(schema.to_string(index=False))

columns = schema["column_name"].tolist()
numeric_types = {
    "TINYINT", "SMALLINT", "INTEGER", "BIGINT", "HUGEINT",
    "UTINYINT", "USMALLINT", "UINTEGER", "UBIGINT",
    "FLOAT", "REAL", "DOUBLE",
    "DECIMAL", "NUMERIC"
}

numeric_columns = [
    row["column_name"]
    for _, row in schema.iterrows()
    if any(
        str(row["column_type"]).upper().startswith(t)
        for t in numeric_types
    )
]

print(f"\nTotal columns: {len(columns)}")
print(f"Numeric columns: {len(numeric_columns)}")


1. SCHEMA
                                     column_name column_type null  key default extra
                                           usdot      BIGINT  YES None    None  None
                                    company_name     VARCHAR  YES None    None  None
                                        dba_name     VARCHAR  YES None    None  None
                                           state     VARCHAR  YES None    None  None
                             has_insurance_event     BOOLEAN  YES None    None  None
                                    event_source     VARCHAR  YES None    None  None
                          event_usdot_matched_by     VARCHAR  YES None    None  None
                                    event_status     VARCHAR  YES None    None  None
                                     policy_line     VARCHAR  YES None    None  None
                                         insurer     VARCHAR  YES None    None  None
                                   insurer_group     V

In [5]:
# --------------------------------------------------
# 2. Sample records
# --------------------------------------------------

print("\n" + "=" * 80)
print("2. FIRST 10 ROWS")
print("=" * 80)

print(con.execute("SELECT * FROM raw LIMIT 10").df().to_string(index=False))


2. FIRST 10 ROWS
 usdot                                company_name                  dba_name state  has_insurance_event event_source event_usdot_matched_by event_status policy_line insurer insurer_group coverage_class form_code class_code policy_number docket_or_certificate_number effective_date expiration_date cancellation_or_end_date end_reason cancellation_form_code filed_or_received_date rejection_reason  event_year years_covered all_insurers_this_usdot_in_event_year  insurer_count_this_usdot_in_event_year  coverage_amount_usd  underlying_limit_usd  min_required_coverage_usd  premium_usd rating_state  vehicle_count insurer_branch agent_name  is_peo  residual_market event_details_json event_company_signature  insurance_event_count  insurance_distinct_insurers insurance_first_event_date insurance_latest_event_date insurance_all_insurers insurance_active_bipd_insurers_as_of_2026_05_26 insurance_active_cargo_insurers_as_of_2026_05_26 insurance_wc_insurers  insurance_cancelled_count  

In [6]:
# --------------------------------------------------
# 3. Total row count
# --------------------------------------------------

print("\n" + "=" * 80)
print("3. TOTAL ROWS")
print("=" * 80)

row_count = con.execute(
    "SELECT COUNT(*) FROM raw"
).fetchone()[0]

print(f"Rows: {row_count:,}")


3. TOTAL ROWS
Rows: 12,243,741


In [7]:
expressions = []

for i, col in enumerate(columns):
    c = quote_ident(col)
    expressions.append(f"COUNT({c}) AS nonnull_{i}")

query = "SELECT " + ", ".join(expressions) + " FROM raw"
result = con.execute(query).fetchone()

profile = []

for i, col in enumerate(columns):
    nonnull = result[i]
    profile.append({
        "column": col,
        "null_count": row_count - nonnull,
        "null_pct": (
            (row_count - nonnull) / row_count * 100
            if row_count else 0
        )
    })

profile_df = pd.DataFrame(profile).sort_values(
    "null_pct", ascending=False
)

print(profile_df.to_string(index=False))

                                          column  null_count   null_pct
                                    co_sic_codes    12243741 100.000000
                                       co_county    12243741 100.000000
                               census_trplimo_16    12243641  99.999183
                              census_mail_barrio    12243601  99.998857
                           census_trpschool_9_15    12243594  99.998799
                            census_trpschool_1_8    12243568  99.998587
                               census_total_cars    12243558  99.998505
                             census_trplimo_9_15    12243482  99.997885
                             census_trpschool_16    12243400  99.997215
                              census_trplimo_1_8    12243384  99.997084
                               census_phy_barrio    12243310  99.996480
                            census_trmschool_1_8    12243202  99.995598
                  census_prior_revoke_dot_number    12243157  99

In [8]:
key_columns = [
    # Company identity
    "usdot",
    "company_name",
    "state",

    # Insurance event details
    "has_insurance_event",
    "event_source",
    "event_usdot_matched_by",
    "event_status",
    "event_year",
    "policy_line",
    "insurer",
    "insurer_group",
    "coverage_class",
    "policy_number",

    # Coverage and policy lifecycle
    "effective_date",
    "expiration_date",
    "cancellation_or_end_date",
    "end_reason",
    "filed_or_received_date",
    "rejection_reason",
    "coverage_amount_usd",
    "min_required_coverage_usd",
    "premium_usd",

    # Insurance history summaries
    "insurance_event_count",
    "insurance_distinct_insurers",
    "insurance_cancelled_count",
    "insurance_replaced_count",
    "insurance_rejected_count",

    # Fleet and safety profile
    "census_power_units",
    "census_total_drivers",
    "crash_count",
    "inspection_count",
    "revocation_count",
    "oos_order_count",

    # Business enrichment
    "co_primary_naics",
    "co_years_in_business",
    "co_employee_band",
    "co_match_confidence",
]

expressions = [
    f"APPROX_COUNT_DISTINCT({quote_ident(col)}) AS {quote_ident(col)}"
    for col in key_columns
]

query = "SELECT " + ", ".join(expressions) + " FROM raw"
print(con.execute(query).fetchdf())

     usdot  company_name  state  has_insurance_event  event_source  \
0  4205801       4222370    117                    2            13   

   event_usdot_matched_by  event_status  event_year  policy_line  insurer  \
0                       3            12          85            9     6453   

   ...  census_power_units  census_total_drivers  crash_count  \
0  ...                2577                  1722          266   

   inspection_count  revocation_count  oos_order_count  co_primary_naics  \
0               932                27                6              1420   

   co_years_in_business  co_employee_band  co_match_confidence  
0                   112                10                    2  

[1 rows x 37 columns]


In [9]:
# --------------------------------------------------
# 5. DuckDB's built-in descriptive summary
# Includes statistics appropriate to each column type
# --------------------------------------------------
print("\n" + "=" * 80)
print("5. DESCRIPTIVE SUMMARY")
print("=" * 80)

summary = con.execute("SUMMARIZE raw").df()
print(summary.to_string(index=False))


5. DESCRIPTIVE SUMMARY
                                     column_name column_type                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        min                         

In [10]:
# --------------------------------------------------
# 6. Numeric statistics
# --------------------------------------------------

if numeric_columns:
    print("\n" + "=" * 80)
    print("6. NUMERIC STATISTICS")
    print("=" * 80)

    expressions = []

    for i, col in enumerate(numeric_columns):
        c = quote_ident(col)

        expressions.extend([
            f"MIN({c}) AS min_{i}",
            f"MAX({c}) AS max_{i}",
            f"AVG(TRY_CAST({c} AS DOUBLE)) AS mean_{i}",
            f"STDDEV_SAMP(TRY_CAST({c} AS DOUBLE)) AS std_{i}",
            f"APPROX_QUANTILE(TRY_CAST({c} AS DOUBLE), 0.25) AS p25_{i}",
            f"APPROX_QUANTILE(TRY_CAST({c} AS DOUBLE), 0.50) AS median_{i}",
            f"APPROX_QUANTILE(TRY_CAST({c} AS DOUBLE), 0.75) AS p75_{i}"
        ])

    values = con.execute(
        "SELECT " + ", ".join(expressions) + " FROM raw"
    ).fetchone()

    numeric_profile = []

    for i, col in enumerate(numeric_columns):
        numeric_profile.append({
            "column": col,
            "min": values[i * 7],
            "max": values[i * 7 + 1],
            "mean": values[i * 7 + 2],
            "std": values[i * 7 + 3],
            "p25": values[i * 7 + 4],
            "median": values[i * 7 + 5],
            "p75": values[i * 7 + 6]
        })

    print(pd.DataFrame(numeric_profile).to_string(index=False))


6. NUMERIC STATISTICS
                                column           min          max          mean          std           p25        median           p75
                                 usdot  1.000000e+00 9.002022e+09  2.141387e+06 4.661490e+06  1.005510e+06  2.025180e+06  3.164565e+06
                            event_year  1.936000e+03 2.027000e+03  2.013818e+03 9.028065e+00  2.007000e+03  2.015421e+03  2.022000e+03
insurer_count_this_usdot_in_event_year  1.000000e+00 4.200000e+01  1.901650e+00 1.569668e+00  1.000000e+00  2.000000e+00  2.000000e+00
                   coverage_amount_usd  0.000000e+00 2.000000e+09  7.593126e+05 1.124660e+06  5.023452e+05  7.500000e+05  1.000000e+06
                  underlying_limit_usd  0.000000e+00 9.285800e+07  1.022706e+04 1.412732e+05  0.000000e+00  0.000000e+00  0.000000e+00
             min_required_coverage_usd  0.000000e+00 1.000000e+07  8.745397e+05 7.346829e+05  7.500000e+05  7.500000e+05  7.500000e+05
                           premi

In [12]:
query = """
WITH values_long AS (
    SELECT 'insurer' AS column_name,
           COALESCE(NULLIF(TRIM(CAST(insurer AS VARCHAR)), ''), '<NULL/BLANK>') AS value
    FROM raw

    UNION ALL
    SELECT 'insurer_group',
           COALESCE(NULLIF(TRIM(CAST(insurer_group AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'event_source',
           COALESCE(NULLIF(TRIM(CAST(event_source AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'event_status',
           COALESCE(NULLIF(TRIM(CAST(event_status AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'policy_line',
           COALESCE(NULLIF(TRIM(CAST(policy_line AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'coverage_class',
           COALESCE(NULLIF(TRIM(CAST(coverage_class AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'form_code',
           COALESCE(NULLIF(TRIM(CAST(form_code AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'class_code',
           COALESCE(NULLIF(TRIM(CAST(class_code AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'rating_state',
           COALESCE(NULLIF(TRIM(CAST(rating_state AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'all_insurers_this_usdot_in_event_year',
           COALESCE(NULLIF(TRIM(CAST(all_insurers_this_usdot_in_event_year AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'insurance_all_insurers',
           COALESCE(NULLIF(TRIM(CAST(insurance_all_insurers AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'insurance_active_bipd_insurers_as_of_2026_05_26',
           COALESCE(NULLIF(TRIM(CAST(insurance_active_bipd_insurers_as_of_2026_05_26 AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'insurance_active_cargo_insurers_as_of_2026_05_26',
           COALESCE(NULLIF(TRIM(CAST(insurance_active_cargo_insurers_as_of_2026_05_26 AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'insurance_wc_insurers',
           COALESCE(NULLIF(TRIM(CAST(insurance_wc_insurers AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'insurance_carriers_by_year',
           COALESCE(NULLIF(TRIM(CAST(insurance_carriers_by_year AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw
),
freq AS (
    SELECT
        column_name,
        value,
        COUNT(*) AS frequency
    FROM values_long
    GROUP BY column_name, value
)
SELECT
    column_name,
    value,
    frequency,
    ROUND(
        100.0 * frequency /
        SUM(frequency) OVER (PARTITION BY column_name),
        3
    ) AS pct_of_rows
FROM freq
ORDER BY column_name, frequency DESC
"""

# Execute using your existing DuckDB connection
result = con.execute(query).fetchdf()

# Display the full distribution
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", 150)

display(result)

RuntimeError: Query interrupted

In [13]:
categorical_columns = [
    "insurer",
    "insurer_group",
    "event_source",
    "event_status",
    "policy_line",
    "coverage_class",
    "form_code",
    "class_code",
    "rating_state",
]

values_clause = ",\n".join(
    f"('{col}', CAST({quote_ident(col)} AS VARCHAR))"
    for col in categorical_columns
)

query = f"""
WITH values_long AS (
    SELECT
        v.column_name,
        COALESCE(
            NULLIF(TRIM(v.value), ''),
            '<NULL/BLANK>'
        ) AS value
    FROM raw
    CROSS JOIN LATERAL (
        VALUES
            {values_clause}
    ) AS v(column_name, value)
),
freq AS (
    SELECT
        column_name,
        value,
        COUNT(*) AS frequency
    FROM values_long
    GROUP BY column_name, value
),
ranked AS (
    SELECT
        column_name,
        value,
        frequency,
        ROUND(
            100.0 * frequency /
            SUM(frequency) OVER (
                PARTITION BY column_name
            ),
            3
        ) AS pct_of_rows,
        ROW_NUMBER() OVER (
            PARTITION BY column_name
            ORDER BY frequency DESC
        ) AS rn
    FROM freq
)
SELECT
    column_name,
    value,
    frequency,
    pct_of_rows
FROM ranked
WHERE rn <= 50
ORDER BY column_name, frequency DESC
"""

result = con.execute(query).fetchdf()

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", 150)

display(result)

,column_name,value,frequency,pct_of_rows
0,class_code,<NULL/BLANK>,5953477,48.625
1,class_code,P,5905367,48.232
2,class_code,E,60480,0.494
3,class_code,5190,19960,0.163
4,class_code,3724,17804,0.145
5,class_code,5606,13566,0.111
6,class_code,7219,12226,0.100
7,class_code,5183,12133,0.099
8,class_code,8810,9713,0.079
9,class_code,5213,9686,0.079


### Discovery queries (for labelling scheme + scoping)

Four follow-up queries to settle open questions before modelling:

- **Q1** — does `end_reason` cleanly split carrier-initiated endings from insured-initiated ones?
- **Q2** — how many distinct DOTs sit in each "current-placement" cohort (Tier A pool size)?
- **Q3** — are there enough short-tenure cancellations to serve as a wrong-fit signal?
- **Q4** — are the `residual_market` flags populated with usable values for validation?


In [11]:
# --------------------------------------------------
# Discovery Q1: end_reason x event_status breakdown
#
# Does end_reason distinguish carrier-initiated non-renewal from
# insured-initiated changes? If so, carrier-initiated endings
# become usable hard-negative labels for the fit model; insured-
# initiated endings should NOT be labelled wrong-fit.
# --------------------------------------------------

query = """
WITH e AS (
    SELECT
        COALESCE(NULLIF(TRIM(CAST(end_reason   AS VARCHAR)), ''), '<NULL>') AS end_reason,
        COALESCE(NULLIF(TRIM(CAST(event_status AS VARCHAR)), ''), '<NULL>') AS event_status
    FROM raw
    WHERE end_reason IS NOT NULL
)
SELECT
    end_reason,
    event_status,
    COUNT(*) AS n,
    ROUND(
        100.0 * COUNT(*)
        / SUM(COUNT(*)) OVER (PARTITION BY end_reason),
        2
    ) AS pct_within_end_reason
FROM e
GROUP BY end_reason, event_status
ORDER BY end_reason, n DESC
"""

pd.set_option("display.max_rows", 500)
display(con.execute(query).fetchdf())


,end_reason,event_status,n,pct_within_end_reason
0,CANC/CANCL,Cancelled,6379,100.00
1,CANCEL,Cancelled,3862192,100.00
2,CANCEL,Name Changed,2,0.00
3,CANCEL,Transferred,2,0.00
4,DUPLICATE,Cancelled,449,100.00
5,NAMECHG,Name Changed,68241,100.00
6,REVOKED,Cancelled,1610,100.00
7,TERM/CANCL,Cancelled,914988,89.87
8,TERM/CANCL,Name Changed,102148,10.03
9,TERM/CANCL,Transferred,1038,0.10


In [12]:
# --------------------------------------------------
# Discovery Q2: Distinct-DOT counts per 'current-placement' field
#
# Establishes how many DOTs we can actually score for mismatch
# (Tier A), vs how many have authority but no active placement
# (Tier B / 'uncovered' candidates we can't trust without
# knowing data completeness).
# --------------------------------------------------

query = """
SELECT 'total_distinct_dots' AS cohort,
       COUNT(DISTINCT usdot) AS n_dots
FROM raw

UNION ALL

SELECT 'has_insurance_event = TRUE',
       COUNT(DISTINCT usdot)
FROM raw
WHERE has_insurance_event = TRUE

UNION ALL

SELECT 'any_insurance_active_bipd_set',
       COUNT(DISTINCT usdot)
FROM raw
WHERE insurance_active_bipd_insurers_as_of_2026_05_26 IS NOT NULL

UNION ALL

SELECT 'any_insurance_active_cargo_set',
       COUNT(DISTINCT usdot)
FROM raw
WHERE insurance_active_cargo_insurers_as_of_2026_05_26 IS NOT NULL

UNION ALL

SELECT 'any_insurance_wc_insurers_set',
       COUNT(DISTINCT usdot)
FROM raw
WHERE insurance_wc_insurers IS NOT NULL

UNION ALL

SELECT 'any_co_wc_carrier_set',
       COUNT(DISTINCT usdot)
FROM raw
WHERE co_wc_carrier IS NOT NULL

UNION ALL

SELECT 'any_active_bipd_OR_cargo_OR_wc',
       COUNT(DISTINCT usdot)
FROM raw
WHERE insurance_active_bipd_insurers_as_of_2026_05_26  IS NOT NULL
   OR insurance_active_cargo_insurers_as_of_2026_05_26 IS NOT NULL
   OR insurance_wc_insurers                            IS NOT NULL

UNION ALL

SELECT 'any_authority_filing',
       COUNT(DISTINCT usdot)
FROM raw
WHERE authority_common_stat   IS NOT NULL
   OR authority_contract_stat IS NOT NULL
   OR authority_broker_stat   IS NOT NULL

UNION ALL

SELECT 'has_authority_but_no_active_bipd',
       COUNT(DISTINCT usdot)
FROM raw
WHERE (authority_common_stat   IS NOT NULL
    OR authority_contract_stat IS NOT NULL
    OR authority_broker_stat   IS NOT NULL)
  AND insurance_active_bipd_insurers_as_of_2026_05_26 IS NULL

ORDER BY n_dots DESC
"""

display(con.execute(query).fetchdf())


,cohort,n_dots
0,total_distinct_dots,4517323
1,any_authority_filing,1679119
2,has_insurance_event = TRUE,1497586
3,has_authority_but_no_active_bipd,1293780
4,any_active_bipd_OR_cargo_OR_wc,461960
5,any_insurance_active_bipd_set,388724
6,any_insurance_wc_insurers_set,70303
7,any_co_wc_carrier_set,70212
8,any_insurance_active_cargo_set,35076


In [13]:
# --------------------------------------------------
# Discovery Q3: Time-to-end distribution for ended policies
#
# Short-tenure endings (<90 days in force) are much more likely
# to indicate wrong-fit (mid-term declines, misrepresentation
# discovered, early non-renewal) than full-term replacements.
# This tells us whether we have a usable short-tenure signal.
# --------------------------------------------------

query = """
WITH t AS (
    SELECT
        DATE_DIFF('day', effective_date, cancellation_or_end_date) AS days_in_force,
        COALESCE(NULLIF(TRIM(CAST(event_status AS VARCHAR)), ''), '<NULL>') AS event_status
    FROM raw
    WHERE effective_date IS NOT NULL
      AND cancellation_or_end_date IS NOT NULL
)
SELECT
    CASE
        WHEN days_in_force < 0    THEN '<0 (data issue)'
        WHEN days_in_force < 30   THEN '00-29 days'
        WHEN days_in_force < 90   THEN '30-89 days'
        WHEN days_in_force < 180  THEN '90-179 days'
        WHEN days_in_force < 365  THEN '180-364 days'
        WHEN days_in_force < 730  THEN '365-729 days'
        ELSE                           '730+ days'
    END AS tenure_bucket,
    event_status,
    COUNT(*) AS n
FROM t
GROUP BY tenure_bucket, event_status
ORDER BY event_status, tenure_bucket
"""

pd.set_option("display.max_rows", 200)
display(con.execute(query).fetchdf())


,tenure_bucket,event_status,n
0,00-29 days,<NULL>,10810
1,180-364 days,<NULL>,6519
2,30-89 days,<NULL>,5380
3,365-729 days,<NULL>,19424
4,730+ days,<NULL>,3276
5,90-179 days,<NULL>,5428
6,<0 (data issue),<NULL>,39
7,180-364 days,Active,1040
8,30-89 days,Active,1018
9,365-729 days,Active,2640


In [14]:
# --------------------------------------------------
# Discovery Q4: residual-market indicators
#
# Where populated, these are hard-labelled wrong-fit signals
# (the DOT couldn't find a voluntary carrier). Confirms whether
# the values distinguish anything useful for validation negatives.
# --------------------------------------------------

query = """
SELECT 'residual_market' AS field,
       CAST(residual_market AS VARCHAR) AS value,
       COUNT(*) AS n
FROM raw
WHERE residual_market IS NOT NULL
GROUP BY 1, 2

UNION ALL

SELECT 'co_wc_residual_market' AS field,
       CAST(co_wc_residual_market AS VARCHAR) AS value,
       COUNT(*) AS n
FROM raw
WHERE co_wc_residual_market IS NOT NULL
GROUP BY 1, 2

ORDER BY field, value
"""

display(con.execute(query).fetchdf())


,field,value,n
0,co_wc_residual_market,false,268684
1,co_wc_residual_market,true,26536
2,residual_market,false,265843
3,residual_market,true,9312


### Mismatch model v3 — binary contrastive fit, 6 policy lines, 4-dimension feature set

**Framing:** binary contrastive. For each `(DOT, carrier, state)` pair, model learns `P(real pair | DOT profile, carrier aggregate profile)`. Observed carrier's rank among all in-state candidates = **appetite outlier score** (reframed from "mismatch" to be honest about PU contamination).

**Grain:** DOT × policy_line × active carrier. Scorable pool is the 462k DOTs with any active placement across the 6 lines.

**Six per-line models (one per policy line):**
- BIPD (auto liability) — ~388k DOTs, pre-computed snapshot
- Cargo — ~35k DOTs, pre-computed snapshot
- WC — ~70k DOTs, pre-computed snapshot
- Broker surety bond — ~30-50k DOTs, event-reconstructed active placement
- Surety (state-level) — ~2-3k DOTs, thin
- GL — ~2-3k DOTs, thin

**Carrier identity:** `(carrier_group, state)` with per-line min-N thresholds (BIPD/Cargo/WC = 20, surety_bond = 15, surety/gl = 10). Rule-based consolidation on top insurer families.

**Carrier aggregates:** from all events in last 24 months (not just currently-active placements), state-partitioned. Captures current appetite, not decade-old patterns.

**Labelling (Option C — clean positives + hard negatives):**
- Positives (y=1): observed placements where `is_residual_market=0 AND is_cancellation_pending=0`
- Hard negatives (y=0): observed placements with `is_residual_market=1` + events with `end_reason='REVOKED'`
- Soft negatives (y=0): K=3 random carriers per positive from same state's pool, anti-joined vs real placements

**Features across 4 fit dimensions** (as features, not label weights):

| dimension | DOT-side | carrier-side (per state) |
|---|---|---|
| appetite | fleet, commodity, hazmat, operation, state, authority, radius | fleet size dist (mean/median/p90), commodity mix, radius mix |
| price proxies | `is_residual_market`, rejected_count, coverage_vs_min, has_layered_coverage, wc_rate_multiplier | — |
| coverage | coverage_amount, authority_min, underlying_limit | — |
| service | — | retention/short-cancel/voluntary-exit/rejection composite |
| timing | days_since_effective, days_until_expiration, years_with_current_carrier | — |

**Baselines:**
- B1: random-in-state
- B2: popularity by state × fleet-size-band

**Validation:**
- Grouped 3-fold CV on `usdot`
- Top-K hit rate (K=1, 5, 10) + MRR per line
- Segmented by state, fleet band
- Long-tenure diagnostic (placements with ≥3 years tenure should score HIGH; replaces residual-market diagnostic since residual-market now trains as hard negative)

**Sub-entity breakdown (Option B — weighted Euclidean distance):** within each recommended group, rank sub-entities by appetite distance to the DOT (not by volume share). Fleet size, risk level, commodity mix, operating radius. Weights derived empirically from within-group variance.

**Output artifacts:**
- Wide CSV: top 10k DOTs × top-20 recommended groups (with best-fit sub-entity per group)
- Long CSV: top-100 full ranking per DOT per line (with `is_observed` flag)

**Compute backend:** LightGBM where available (2-3× faster); falls back to `HistGradientBoostingClassifier`.

**Known limitations (methodology section):**
- PU contamination — positives contain inertia placements we can't identify; output reframed as "appetite outliers"
- No ground truth for mismatch — primary validation is ranking consistency vs baselines, not direct
- Rosella appointment list unknown — some top-K recommendations may name carriers Rosella can't appoint
- Agent/channel effects confounded with appetite (99.9% null)
- Snapshot staleness (May 2026 vs Oct 2026)
- Service composite is a behavioural proxy, not measured service quality
- Price-proxy features are weak (premium 98.4% null)
- MCS-150 fleet attributes can be stale
- No carrier state licensing data — approximated by "has recent placements in state"
- Carrier state footprint changes only partially handled by 24-month filter (recent entries may be dropped, recent exits may be falsely recommended)
- Hand-tuned priority formula (fleet weight = 0.1)


In [13]:
# ============================================================
# v3 Part 1A — Imports, config, logging utilities
# ============================================================
# Run order dependency: this cell must run before everything else in v3.

import re
import gc
import time
import numpy as np
import pandas as pd
from datetime import date
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.ensemble import HistGradientBoostingClassifier

# Prefer LightGBM when available (2-3x faster CPU); HGBT fallback.
try:
    import lightgbm as lgb
    _MODEL_BACKEND = 'lightgbm'
except ImportError:
    _MODEL_BACKEND = 'sklearn'
print(f"Model backend: {_MODEL_BACKEND}")

# Elapsed-time logger — call _log() at every stage boundary
_T0 = time.time()
def _elapsed():
    return f"{time.time() - _T0:.1f}s"
def _log(msg):
    print(f"[t={_elapsed()}] {msg}")

# Memory logger
try:
    import psutil
    _proc = psutil.Process()
    def mem_gb(): return _proc.memory_info().rss / 1024**3
except ImportError:
    def mem_gb(): return 0.0
def _log_mem(label):
    print(f"  [mem] {label}: {mem_gb():.2f} GB")

# Config constants
RANDOM_SEED              = 42
N_NEGATIVES_PER_POSITIVE = 3
RECENCY_WINDOW_MONTHS    = 24
TOP_N_OUTPUT             = 10_000
TOP_K_PER_DOT            = 20
TOP_K_FULL_RANKING       = 100

MIN_CARRIER_PLACEMENTS = {
    'bipd': 20, 'cargo': 20, 'wc': 20,
    'surety_bond': 15, 'surety': 10, 'gl': 10,
}
MIN_SUB_ENTITY_PLACEMENTS = 10
# Carriers with >= this share of residual-market events are excluded from the
# recommendation pool. They remain in training as hard negatives, so the model
# still learns to avoid them for voluntary-market DOTs.
RESIDUAL_MARKET_CARRIER_THRESHOLD = 0.5

APPETITE_DIST_FEATURES = [
    'power_units_log', 'crash_count', 'inspection_oos_viol',
    'crgo_genfreight', 'crgo_bldgmat', 'crgo_livestock',
    'crgo_oilfield', 'crgo_intermodal', 'crgo_chem', 'crgo_passengers',
]

LINES_IN_SCOPE = ['auto liability', 'auto', 'cargo', 'wc',
                  'broker surety bond', 'surety', 'gl']

SNAPSHOT_DATE = pd.Timestamp('2026-05-26')  # from insurance_active_bipd_insurers_as_of_...

rng = np.random.default_rng(RANDOM_SEED)
_log("Part 1A complete — config loaded")


Model backend: lightgbm
[t=0.0s] Part 1A complete — config loaded


In [14]:
# ============================================================
# v3 Part 1B — Carrier consolidation rules (tiered)
# ============================================================
# 78 rules organized by confidence tier. See `rule_vs_provider_disagreements.csv`
# (written in Part 1C) and `carrier_consolidation_report.csv` (Part 2A) to spot-check.
_log("starting Part 1B")

CARRIER_RULES = [
    # ============================================================
    # 3-tuples: (regex, group_name, confidence_tier)
    #
    # Tier 1 — High confidence. Industry-well-known structure, regex is
    #          specific. Safe to trust.
    # Tier 2 — Moderate confidence. Known acquisitions / documented
    #          sub-entities. Verified from training data but not web.
    # Tier 3 — Lower confidence. Regex may over-match, OR sub-entity
    #          ownership details are less certain. VERIFY.
    # Tier 4 — Speculative. Added without careful verification of regex
    #          specificity or exact sub-entity ownership. VERIFY.
    #
    # User action: Review Tier 3/4 rules in `rule_vs_provider_disagreements.csv`
    # (runtime diagnostic). Where the data provider disagrees with our rule,
    # the provider's answer typically wins (edit CARRIER_RULES accordingly).
    # ============================================================

    # ================= TIER 1 — HIGH CONFIDENCE =================
    # Progressive Corporation family (well-documented commercial trucking book)
    (r'PROGRESSIVE',              'PROGRESSIVE GRP', 1),
    (r'ARTISAN\s*&?\s*TRUCKERS',  'PROGRESSIVE GRP', 1),
    (r'DRIVE NEW JERSEY',         'PROGRESSIVE GRP', 1),
    (r'UNITED FINANCIAL CASUALTY','PROGRESSIVE GRP', 1),
    (r'NATIONAL CONTINENTAL',     'PROGRESSIVE GRP', 1),

    # Berkshire Hathaway core (NICO + NLF + Homestate + Direct + GEICO Marine)
    (r'BERKSHIRE HATHAWAY',       'BERKSHIRE HATHAWAY GRP', 1),
    (r'NATIONAL INDEMNITY',       'BERKSHIRE HATHAWAY GRP', 1),
    (r'NATIONAL LIABILITY',       'BERKSHIRE HATHAWAY GRP', 1),
    (r'GEICO MARINE',             'BERKSHIRE HATHAWAY GRP', 1),

    # Canal Insurance Group
    (r'CANAL',                    'CANAL GRP', 1),

    # Travelers (incl. Northland commercial trucking book, acquired 2001)
    (r'NORTHLAND',                'TRAVELERS GRP', 1),
    (r'TRAVELERS',                'TRAVELERS GRP', 1),

    # Zurich North America
    (r'ZURICH',                   'ZURICH INS GRP', 1),
    (r'EMPIRE FIRE',              'ZURICH INS GRP', 1),

    # Standalone/well-known trucking carriers
    (r'GREAT WEST CASUALTY',      'GREAT WEST', 1),
    (r'HALLMARK',                 'HALLMARK GRP', 1),
    (r'OOIDA',                    'OOIDA RRG', 1),
    # NOTE: removed broad LLOYD'?S rule. In Texas, "Lloyds" is a corporate
    # structure (not actual Lloyd's of London), so e.g. HARTFORD LLOYDS INS CO
    # is a Hartford subsidiary, not Lloyd's. Those are better left to the
    # provider map / raw fallback.
    (r'UNDERWRITERS AT LLOYD',    'LLOYDS LONDON', 1),
    (r'SENTRY',                   'SENTRY INS GRP', 1),
    (r'TRISURA',                  'TRISURA', 1),

    # IAT main (Occidental Fire + explicit Harco)
    (r'OCCIDENTAL FIRE',          'IAT GRP', 1),

    # AmTrust main brand + Wesco (its main US arm)
    (r'WESCO',                    'AMTRUST NGH GRP', 1),
    (r'AMTRUST',                  'AMTRUST NGH GRP', 1),

    # W.R. Berkley (Carolina Casualty is well-known subsidiary)
    (r'CAROLINA CASUALTY',        'WR BERKLEY CORP GRP', 1),
    (r'W\.?\s*R\.?\s*BERKLEY',    'WR BERKLEY CORP GRP', 1),

    # Fairfax Financial (Hudson + Crum & Forster are well-known subsidiaries)
    (r'HUDSON',                   'FAIRFAX FIN GRP', 1),
    (r'CRUM.{0,3}FORSTER',        'FAIRFAX FIN GRP', 1),

    # American Financial Group / Great American + National Interstate (AFG subsidiaries)
    (r'NATIONAL INTERSTATE',      'AMERICAN FINANCIAL GRP', 1),
    (r'GREAT AMERICAN',           'AMERICAN FINANCIAL GRP', 1),

    # Kemper (rebranded from Unitrin in 2011)
    (r'UNITRIN',                  'KEMPER GRP', 1),
    (r'KEMPER',                   'KEMPER GRP', 1),

    # Specialty standalone
    (r'PACIFIC FINANCIAL',        'PACIFIC FIN', 1),
    (r'LINCOLN GENERAL',          'LINCOLN GENERAL', 1),

    # Nationwide core (National Casualty is Nationwide subsidiary)
    # Anchored so "OMAHA NATIONAL CASUALTY COMPANY" (independent Nebraska
    # insurer) doesn't collapse into Nationwide.
    (r'^NATIONAL CASUALTY',       'NATIONWIDE CORP GRP', 1),
    (r'NATIONWIDE',               'NATIONWIDE CORP GRP', 1),

    # Allstate core
    (r'ALLSTATE',                 'ALLSTATE GRP', 1),

    # Liberty Mutual core
    # NOTE: removed LIBERTY INS catch-all. It matched e.g. AMERICAN LIBERTY
    # INS CO (actually Benchmark Holdings). Specific Liberty subs that provider
    # doesn't map will fall through to raw-fallback.
    (r'LIBERTY MUTUAL',           'LIBERTY MUT GRP', 1),

    # State WC funds
    (r'TEXAS MUTUAL',             'TEXAS MUTUAL INS CO', 1),
    (r'PINNACOL',                 'PINNACOL ASSURANCE', 1),
    (r'STATE FUND',               'STATE FUND', 1),

    # Hartford — anchored so "NATIONAL FIRE INS CO OF HARTFORD" (actually
    # a CNA subsidiary) doesn't get swept in by the trailing-HARTFORD match.
    (r'^HARTFORD\b',              'HARTFORD FIRE & CAS GRP', 1),

    # State Auto — removed; Liberty Mutual acquired State Auto Financial in
    # 2022, so provider map (which routes to LIBERTY MUT GRP) is correct now.

    # Independent / well-known standalone
    (r'ACUITY',                   'ACUITY', 1),
    (r'AUTO.{0,2}OWNERS',         'AUTO OWNERS GRP', 1),
    (r'CLEAR BLUE',               'CLEAR BLUE', 1),
    (r'QUALITAS',                 'QUALITAS', 1),
    (r'DB INSURANCE',             'DB INS GRP', 1),
    (r'MERITZ',                   'MERITZ GRP', 1),

    # Builders Mutual — restricted so later T4 rules can match
    # BUILDERS INS / MIDWEST BUILDERS (which route to their own groups)
    (r'BUILDERS MUT',             'BUILDERS GRP', 1),

    # ================= TIER 2 — MODERATE CONFIDENCE =================
    # Known acquisitions, documented sub-entities

    # IAT subsidiaries beyond Occidental Fire
    (r'WILSHIRE',                 'IAT GRP', 2),
    (r'HARCO NATIONAL',           'IAT GRP', 2),

    # Lancer → Core Specialty (2022 acquisition)
    (r'LANCER',                   'CORE SPECIALTY GRP', 2),

    # Markel / State National (Markel acquired State National 2017)
    (r'UNITED SPECIALTY',         'MARKEL CORP GRP', 2),
    (r'STATE NATIONAL',           'MARKEL CORP GRP', 2),
    (r'MARKEL',                   'MARKEL CORP GRP', 2),

    # Accredited (post Randall & Quilter spinoff; now Onex-owned)
    (r'ACCREDITED',               'RANDALL & QUILTER INVESTMENT GRP', 2),

    # National General / Integon (Allstate acquired 2021 — kept separate for appetite distinction)
    (r'INTEGON',                  'NATIONAL GENERAL / INTEGON', 2),
    (r'NATIONAL GENERAL',         'NATIONAL GENERAL / INTEGON', 2),

    # Nationwide E&S arm
    (r'SCOTTSDALE',               'NATIONWIDE CORP GRP', 2),

    # Liberty Mutual acquisitions
    (r'SAFECO',                   'LIBERTY MUT GRP', 2),   # acquired 2008
    (r'IRONSHORE',                'LIBERTY MUT GRP', 2),   # acquired 2017

    # Chubb (post ACE merger 2016; Federal Insurance is Chubb subsidiary)
    (r'ACE AMERICAN',             'CHUBB LTD GRP', 2),
    (r'ACE PROPERTY',             'CHUBB LTD GRP', 2),
    (r'CHUBB',                    'CHUBB LTD GRP', 2),
    (r'FEDERAL INSURANCE',        'CHUBB LTD GRP', 2),

    # CNA (Continental Casualty + Transportation Ins are CNA subsidiaries)
    (r'\bCNA\b',                 'CNA INS GRP', 2),
    (r'CONTINENTAL CASUALTY',     'CNA INS GRP', 2),
    (r'TRANSPORTATION INS',       'CNA INS GRP', 2),

    # Everest Re
    (r'EVEREST',                  'EVEREST REINS HOLDINGS GRP', 2),

    # Allstate acquisitions
    (r'ENCOMPASS',                'ALLSTATE GRP', 2),
    (r'ESURANCE',                 'ALLSTATE GRP', 2),

    # Nationwide acquisitions
    (r'HARLEYSVILLE',             'NATIONWIDE CORP GRP', 2),   # acquired 2012

    # Baldwin & Lyons / Protective / Sagamore — provider still groups these
    # under BALDWIN & LYONS GRP (not Progressive), so follow provider.
    (r'PROTECTIVE INSURANCE',     'BALDWIN & LYONS GRP', 2),
    (r'BALDWIN.{0,3}LYONS',       'BALDWIN & LYONS GRP', 2),
    (r'SAGAMORE INS',             'BALDWIN & LYONS GRP', 2),

    # ================= TIER 3 — LOWER CONFIDENCE (verify) =================
    # Regex may over-match OR sub-entity ownership less certain

    # Berkshire — Columbia Insurance may collide with Columbia Mutual (independent)
    (r'COLUMBIA INSURANCE',       'BERKSHIRE HATHAWAY GRP', 3),
    (r'GUARD INSURANCE',          'BERKSHIRE HATHAWAY GRP', 3),

    # Fairfax — Zenith, Allied World
    (r'ZENITH',                   'FAIRFAX FIN GRP', 3),
    (r'ALLIED WORLD',             'FAIRFAX FIN GRP', 3),

    # Nationwide — "Allied" could over-match
    (r'ALLIED PROPERTY',          'NATIONWIDE CORP GRP', 3),

    # Hanover subsidiary
    (r'MASSACHUSETTS BAY INS',    'THE HANOVER INS GRP', 3),

    # Argo Group (Colony Specialty is Argo E&S)
    (r'COLONY (SPECIALTY|INS)',   'ARGONAUT GRP', 3),

    # Tokio Marine subsidiaries
    (r'PHILADELPHIA (CONSOL|INDEM|INS)', 'TOKIO MARINE HOLDINGS INC GRP', 3),
    (r'HOUSTON CASUALTY',         'TOKIO MARINE HOLDINGS INC GRP', 3),

    # AIG subsidiaries (specific sub-entity names)
    (r'LEXINGTON INSURANCE',      'AMERICAN INTL GRP', 3),
    (r'NATIONAL UNION FIRE',      'AMERICAN INTL GRP', 3),
    (r'NEW HAMPSHIRE INS',        'AMERICAN INTL GRP', 3),
    (r'\bAIG\b',                  'AMERICAN INTL GRP', 3),
    (r'AMERICAN INTERNATIONAL',   'AMERICAN INTL GRP', 3),

    # Old Republic subsidiary
    (r'BITCO INS',                'OLD REPUBLIC GRP', 3),

    # XL / AXA XL subsidiaries
    (r'INDIAN HARBOR',            'XL AMER GRP', 3),
    (r'GREENWICH INS',            'XL AMER GRP', 3),

    # Other sub-entities
    (r'EXPLORER INS',             'ICW GRP ASSETS INC GRP', 3),
    (r'MT\.? HAWLEY',              'RLI INS GRP', 3),
    (r'ENDURANCE (SPECIALTY|ASSURANCE)', 'SOMPO GRP', 3),
    (r'ATLANTIC STATES INS',      'DONEGAL GRP', 3),

    # ================= TIER 4 — SPECULATIVE (verify) =================
    # Added in bulk based on data-provider insurer_group values; regex specificity
    # and exact sub-entity membership not individually verified

    # Erie
    (r'ERIE INSURANCE',           'ERIE INS GRP', 4),
    (r'ERIE INDEMNITY',           'ERIE INS GRP', 4),

    # Cincinnati Financial
    (r'CINCINNATI INSURANCE',     'CINCINNATI FIN GRP', 4),
    (r'CINCINNATI CASUALTY',      'CINCINNATI FIN GRP', 4),
    (r'CINCINNATI SPECIALTY',     'CINCINNATI FIN GRP', 4),

    # Selective
    (r'SELECTIVE',                'SELECTIVE INS GRP', 4),

    # Federated Mutual
    (r'FEDERATED MUTUAL',         'FEDERATED MUT GRP', 4),
    (r'FEDERATED SERVICE',        'FEDERATED MUT GRP', 4),

    # FCCI
    (r'FCCI',                     'FCCI MUT INS GRP', 4),

    # EMC Insurance
    (r'EMC INSURANCE',            'EMC INS CO GRP', 4),
    (r'EMC PROPERTY',             'EMC INS CO GRP', 4),
    (r'EMCASCO',                  'EMC INS CO GRP', 4),

    # Donegal
    (r'DONEGAL',                  'DONEGAL GRP', 4),

    # ProAssurance
    (r'PROASSURANCE',             'PROASSURANCE CORP GRP', 4),

    # Pennsylvania National
    (r'PENNSYLVANIA NATIONAL',    'PENNSYLVANIA NATL INS GRP', 4),

    # Tokio Marine parent name match
    (r'TOKIO MARINE',             'TOKIO MARINE HOLDINGS INC GRP', 4),

    # QBE
    (r'\bQBE\b',                  'QBE INS GRP', 4),

    # Starr Companies
    (r'STARR INDEMNITY',          'STARR GRP', 4),
    (r'STARR SPECIALTY',          'STARR GRP', 4),
    (r'STARR SURPLUS',            'STARR GRP', 4),

    # Argo parent name match
    (r'ARGONAUT',                 'ARGONAUT GRP', 4),

    # Hanover parent name match
    (r'HANOVER INSURANCE',        'THE HANOVER INS GRP', 4),

    # Old Republic parent name match
    (r'OLD REPUBLIC',             'OLD REPUBLIC GRP', 4),

    # AmeriSure
    (r'AMERISURE',                'AMERISURE CO GRP', 4),

    # Arch
    (r'ARCH INSURANCE',           'ARCH INS GRP', 4),
    (r'ARCH SPECIALTY',           'ARCH INS GRP', 4),

    # AmeriSafe
    (r'AMERISAFE',                'AMERISAFE GRP', 4),

    # XL parent name matches
    (r'XL INSURANCE',             'XL AMER GRP', 4),
    (r'XL SPECIALTY',             'XL AMER GRP', 4),
    (r'AXA XL',                   'XL AMER GRP', 4),

    # ICW parent name match
    (r'INSURANCE COMPANY OF THE WEST', 'ICW GRP ASSETS INC GRP', 4),

    # Mercury
    (r'MERCURY (GENERAL|INSURANCE|CASUALTY|INDEMNITY)', 'MERCURY GRP', 4),

    # Independents (standalone names)
    (r'GRINNELL',                 'GRINNELL MUT GRP', 4),
    (r'PEKIN',                    'PEKIN INS GRP', 4),
    (r'HISCOX',                   'HISCOX', 4),

    # RLI parent name match
    (r'\bRLI\b',                  'RLI INS GRP', 4),

    # Sompo parent name match
    (r'SOMPO',                    'SOMPO GRP', 4),

    # AXIS
    (r'\bAXIS\b (INSURANCE|SPECIALTY|SURPLUS)', 'AXIS CAPITAL', 4),

    # Specialty / WC funds
    (r'MAINE EMPLOYERS',          'MAINE EMPLOYERS MUT INS GRP', 4),
    (r'WORKERS COMP FUND',        'WORKERS COMP FUND GRP', 4),
    (r'AMERICAN CONTRACTORS',     'AMERICAN CONTRACTORS INS GRP', 4),
    (r'MIDWEST BUILDERS',         'MIDWEST BUILDERS CAS GRP', 4),
    # NOTE: removed BUILDERS INS catch-all — matched Utah / National / Nevada
    # entities that are independent of The Builders Insurance Group of Georgia.

    # Group 1001
    (r'GROUP ONE THOUSAND',       'GROUP ONE THOUSAND ONE GRP', 4),

    # Summitpoint
    (r'SUMMITPOINT',              'SUMMITPOINT INSURANCE CO', 4),

    # Mapfre
    (r'MAPFRE',                   'MAPFRE GRP', 4),

    # Americas Insurance
    (r'AMERICAS INSURANCE',       'AMERICAS INS', 4),
]

CARRIER_RULES_C = [(re.compile(p), g, t) for p, g, t in CARRIER_RULES]
_MULTI_SPLIT    = re.compile(r'[|;]|,(?=\s*[A-Z0-9])')

def consolidate_multi(raw):
    """insurance_active_* can carry multiple carriers in one string.
    Split on separators and consolidate each. Returns deduped list."""
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return []
    out, seen = [], set()
    for part in _MULTI_SPLIT.split(str(raw)):
        g = consolidate_one(part)  # hybrid: provider map + rules (defined in Part 1C)
        if g and g not in seen:
            seen.add(g)
            out.append(g)
    return out

# Pure-rules consolidation (used by consolidate_multi before DATA_PROVIDER_MAP exists)
def _consolidate_one_raw(raw):
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return None
    s = str(raw).strip().upper()
    if not s or s in ('NULL', 'NONE', 'N/A'):
        return None
    for pat, grp, _tier in CARRIER_RULES_C:
        if pat.search(s):
            return grp
    return s  # raw-fallback

_log(f"Part 1B complete — {len(CARRIER_RULES)} rules compiled")


[t=3.6s] starting Part 1B
[t=3.6s] Part 1B complete — 145 rules compiled


In [15]:
# ============================================================
# v3 Part 1C — Data vintage + Hybrid consolidation + Diagnostic
# ============================================================
# Dynamic data vintage (fixes the "today's date" bug) + DATA_PROVIDER_MAP from
# the populated 2.7% of insurer_group rows + runtime disagreement check.
_log("starting Part 1C")

# ------------------------------------------------------------
# Data vintage: use the file's actual event date range, not today.
# Raw CSV has entries like 9001-01-01 (data entry errors), so clip
# before taking MIN/MAX. Past-only columns (effective_date, crash
# dates, etc.) are capped at CURRENT_DATE; a 2028 effective_date is
# as much of a data error as a 9001 one. Policy *expiration* /
# scheduled-cancellation dates can legitimately sit in the future,
# so those get a + 2 year buffer in cells 27 and 30.
# ------------------------------------------------------------
_VALID_DATE_LO_SQL      = "DATE '1990-01-01'"
_VALID_DATE_HI_PAST_SQL = "CURRENT_DATE"
# Future-allowed cap is a hard "obviously-a-data-error" ceiling at year 2100;
# multi-decade surety bonds / long-term contracts (even 50 years out) are kept.
_VALID_DATE_HI_FUT_SQL  = "DATE '2100-01-01'"
_VALID_DATE_LO          = pd.Timestamp('1990-01-01')
_VALID_DATE_HI_PAST     = pd.Timestamp.today().normalize()
_VALID_DATE_HI_FUT      = pd.Timestamp('2100-01-01')

_vintage_row = con.execute(f"""
    SELECT MIN(effective_date) AS min_d,
           MAX(effective_date) AS max_d
    FROM raw
    WHERE effective_date IS NOT NULL
      AND effective_date BETWEEN {_VALID_DATE_LO_SQL} AND {_VALID_DATE_HI_PAST_SQL}
""").fetchone()
_DATA_VINTAGE  = pd.Timestamp(_vintage_row[1])
_EARLIEST_DATE = pd.Timestamp(_vintage_row[0])
_WINDOW_START  = _DATA_VINTAGE - pd.DateOffset(months=RECENCY_WINDOW_MONTHS)
print(f"  data event-date range (sanitized): {_EARLIEST_DATE.date()} → {_DATA_VINTAGE.date()}")
print(f"  24-month window: {_WINDOW_START.date()} → {_DATA_VINTAGE.date()}")
print(f"  snapshot column date: {SNAPSHOT_DATE.date()}")

# Report any raw effective_date values that fell outside the valid window.
_outlier_rows = con.execute(f"""
    SELECT COUNT(*) AS n,
           MIN(effective_date) AS lo,
           MAX(effective_date) AS hi
    FROM raw
    WHERE effective_date IS NOT NULL
      AND (effective_date < {_VALID_DATE_LO_SQL} OR effective_date > {_VALID_DATE_HI_PAST_SQL})
""").fetchone()
if _outlier_rows[0] and _outlier_rows[0] > 0:
    print(f"  WARNING: {_outlier_rows[0]:,} effective_date values outside "
          f"[1990, today] (min={_outlier_rows[1]}, max={_outlier_rows[2]}) — "
          f"excluded from vintage computation")

# Convert to SQL date literals for subsequent queries
_VINTAGE_SQL      = f"DATE '{_DATA_VINTAGE.date()}'"
_WINDOW_START_SQL = f"DATE '{_WINDOW_START.date()}'"
_SNAPSHOT_SQL     = f"DATE '{SNAPSHOT_DATE.date()}'"

# ------------------------------------------------------------
# Build data-provider raw_insurer -> group mapping from 2.7% populated rows
# ------------------------------------------------------------
print("\nBuilding data-provider insurer_group mapping...")
_provider_df = con.execute("""
    WITH votes AS (
        SELECT UPPER(TRIM(CAST(insurer AS VARCHAR))) AS insurer_raw,
               UPPER(TRIM(CAST(insurer_group AS VARCHAR))) AS provider_group,
               COUNT(*) AS n
        FROM raw
        WHERE insurer IS NOT NULL
          AND insurer_group IS NOT NULL
          AND TRIM(CAST(insurer_group AS VARCHAR)) <> ''
        GROUP BY 1, 2
    ),
    ranked AS (
        SELECT insurer_raw, provider_group,
               ROW_NUMBER() OVER (PARTITION BY insurer_raw ORDER BY n DESC) AS rn
        FROM votes
    )
    SELECT insurer_raw, provider_group FROM ranked WHERE rn = 1
""").df()
DATA_PROVIDER_MAP = dict(zip(_provider_df['insurer_raw'], _provider_df['provider_group']))
print(f"  {len(DATA_PROVIDER_MAP):,} raw_insurer -> group mappings from data provider")

# ------------------------------------------------------------
# Hybrid consolidate_one: provider map first (authoritative — they have the
# most recent snapshot of carrier ownership), then rules, then raw fallback.
# If provider stored the raw name as the group, that's still an explicit
# decision from the data provider to keep that insurer standalone.
# ------------------------------------------------------------
def consolidate_one(raw):
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return None
    s = str(raw).strip().upper()
    if not s or s in ('NULL', 'NONE', 'N/A'):
        return None
    g = DATA_PROVIDER_MAP.get(s)
    if g: return g
    for pat, grp, _tier in CARRIER_RULES_C:
        if pat.search(s):
            return grp
    return s

# ------------------------------------------------------------
# Rule-vs-provider disagreement diagnostic
# ------------------------------------------------------------
def _apply_rules_only(raw):
    if raw is None: return (None, None)
    s = str(raw).strip().upper()
    if not s: return (None, None)
    for pat, grp, tier in CARRIER_RULES_C:
        if pat.search(s):
            return (grp, tier)
    return (None, None)

print("\nRule-vs-provider disagreement diagnostic...")
_disagreements = []
for _ins_raw, _prov_grp in DATA_PROVIDER_MAP.items():
    _rule_grp, _rule_tier = _apply_rules_only(_ins_raw)
    if _rule_grp is not None and _rule_grp != _prov_grp:
        _disagreements.append({'insurer_raw': _ins_raw, 'rule_output': _rule_grp,
                               'rule_tier': _rule_tier, 'provider_output': _prov_grp})

if _disagreements:
    _disagree_df = pd.DataFrame(_disagreements).sort_values(['rule_tier', 'rule_output'])
    _disagree_df.to_csv('rule_vs_provider_disagreements.csv', index=False)
    print(f"  Found {len(_disagree_df):,} disagreements -> rule_vs_provider_disagreements.csv")
    print(f"  Breakdown by tier: {_disagree_df['rule_tier'].value_counts().to_dict()}")
    print("\n  First 20 disagreements:")
    print(_disagree_df.head(20).to_string(index=False))
else:
    print("  No disagreements — rules agree with data provider on all overlapping insurers.")

_log("Part 1C complete")


[t=9.3s] starting Part 1C
  data event-date range (sanitized): 1990-01-01 → 2026-10-22
  24-month window: 2024-10-22 → 2026-10-22
  snapshot column date: 2026-05-26

Building data-provider insurer_group mapping...
  773 raw_insurer -> group mappings from data provider

Rule-vs-provider disagreement diagnostic...
  Found 35 disagreements -> rule_vs_provider_disagreements.csv
  Breakdown by tier: {1: 17, 4: 10, 2: 7, 3: 1}

  First 20 disagreements:
                             insurer_raw             rule_output  rule_tier                          provider_output
                  ACUITY A MUTUAL INS CO                  ACUITY          1                   ACUITY A MUTUAL INS CO
                AMTRUST INS CO OF KS INC             AMTRUST GRP          1                          AMTRUST NGH GRP
                            WESCO INS CO             AMTRUST GRP          1                          AMTRUST NGH GRP
        NATIONAL FIRE INS CO OF HARTFORD HARTFORD FIRE & CAS GRP          1     

In [ ]:
# ============================================================
# v3 Part 1D — Sanity check (windowed properly to <= SNAPSHOT_DATE)
# ============================================================
# Fixes earlier bug: previously the event-derived "latest Active" could be AFTER
# the May 2026 snapshot, producing false disagreements. Now filtered at the
# snapshot date for apples-to-apples comparison.
_log("starting Part 1D (sanity check)")

print("=" * 70)
print(f"SANITY CHECK: active_bipd snapshot ({SNAPSHOT_DATE.date()}) vs latest Active-event")
print("                                   filtered to events on or before snapshot")
print("=" * 70)

sanity_q = f"""
WITH active_bipd_events AS (
    SELECT usdot,
           UPPER(TRIM(CAST(insurer AS VARCHAR))) AS event_insurer,
           effective_date,
           ROW_NUMBER() OVER (
               PARTITION BY usdot
               ORDER BY COALESCE(effective_date, filed_or_received_date) DESC NULLS LAST
           ) AS rn
    FROM raw
    WHERE LOWER(TRIM(CAST(policy_line AS VARCHAR))) IN ('auto liability', 'auto')
      AND LOWER(TRIM(CAST(event_status AS VARCHAR))) IN ('active', 'active (cancellation pending)')
      AND insurer IS NOT NULL
      AND effective_date <= {_SNAPSHOT_SQL}       -- windowed to snapshot date
),
latest_event AS (
    SELECT usdot, event_insurer, effective_date FROM active_bipd_events WHERE rn = 1
),
dot_snapshot AS (
    SELECT DISTINCT usdot,
           UPPER(TRIM(CAST(insurance_active_bipd_insurers_as_of_2026_05_26 AS VARCHAR))) AS snapshot_insurer
    FROM raw
    WHERE insurance_active_bipd_insurers_as_of_2026_05_26 IS NOT NULL
)
SELECT CASE
         WHEN snapshot_insurer IS NULL AND event_insurer IS NULL THEN 'both_null'
         WHEN snapshot_insurer IS NOT NULL AND event_insurer IS NULL THEN 'snapshot_only (no Active event found)'
         WHEN snapshot_insurer IS NULL AND event_insurer IS NOT NULL THEN 'event_only (not in snapshot)'
         WHEN snapshot_insurer = event_insurer THEN 'exact_match'
         WHEN POSITION(event_insurer IN snapshot_insurer) > 0
           OR POSITION(snapshot_insurer IN event_insurer) > 0 THEN 'partial_match'
         ELSE 'disagreement'
       END AS category,
       COUNT(*) AS n_dots
FROM dot_snapshot
FULL OUTER JOIN latest_event USING (usdot)
WHERE snapshot_insurer IS NOT NULL OR event_insurer IS NOT NULL
GROUP BY 1
ORDER BY n_dots DESC
"""
sanity_df = con.execute(sanity_q).fetchdf()
print(sanity_df.to_string(index=False))
total = sanity_df['n_dots'].sum()
agree = sanity_df.loc[sanity_df['category'].isin(['exact_match', 'partial_match']), 'n_dots'].sum()
if total > 0:
    print(f"\nAgreement rate: {agree}/{total} = {100*agree/total:.1f}%")

_log("Part 1D complete")


In [ ]:
# ============================================================
# v3 Part 1E — Build dot_features table (one row per DOT)
# ============================================================
# Changes vs previous:
# - state fields UPPER(TRIM()) normalized
# - is_residual_market / is_cancellation_pending restricted to events within recency window
# - insurance_history_years clipped to >= 0 (handled in Part 2 derivation; not here)
_log("starting Part 1E — building dot_features")

con.execute("DROP TABLE IF EXISTS dot_features")
con.execute(f"""
CREATE TABLE dot_features AS
SELECT
    usdot,
    ANY_VALUE(company_name)             AS company_name,
    UPPER(TRIM(ANY_VALUE(state)))           AS state,
    UPPER(TRIM(ANY_VALUE(census_phy_state))) AS phy_state,

    -- fleet shape
    MAX(census_power_units)             AS power_units,
    MAX(census_total_drivers)           AS total_drivers,
    MAX(census_total_cdl)               AS total_cdl,
    MAX(census_truck_units)             AS truck_units,
    ANY_VALUE(census_fleetsize)         AS fleetsize_band,

    -- operation
    ANY_VALUE(census_carrier_operation) AS carrier_operation,
    ANY_VALUE(census_classdef)          AS classdef,
    ANY_VALUE(census_hm_ind)            AS hazmat_ind,
    MAX(census_mcs150_mileage)          AS mcs150_mileage,

    -- radius
    MAX(census_interstate_beyond_100_miles) AS inter_beyond_100,
    MAX(census_interstate_within_100_miles) AS inter_within_100,
    MAX(census_intrastate_beyond_100_miles) AS intra_beyond_100,
    MAX(census_intrastate_within_100_miles) AS intra_within_100,

    -- authority
    ANY_VALUE(authority_common_stat)    AS authority_common_stat,
    ANY_VALUE(authority_contract_stat)  AS authority_contract_stat,
    ANY_VALUE(authority_broker_stat)    AS authority_broker_stat,
    ANY_VALUE(authority_bipd_file)      AS authority_bipd_file,
    ANY_VALUE(authority_cargo_file)     AS authority_cargo_file,
    ANY_VALUE(authority_bond_file)      AS authority_bond_file,
    CASE WHEN ANY_VALUE(authority_common_stat)   IS NOT NULL
           OR ANY_VALUE(authority_contract_stat) IS NOT NULL
           OR ANY_VALUE(authority_broker_stat)   IS NOT NULL
         THEN 1 ELSE 0 END              AS has_authority,
    MAX(TRY_CAST(authority_min_cov_amount AS DOUBLE)) AS authority_min_cov_amount,

    -- commodity flags (structural-zero null regime)
    MAX(CASE WHEN TRIM(COALESCE(census_crgo_genfreight, '')) <> '' THEN 1 ELSE 0 END) AS crgo_genfreight,
    MAX(CASE WHEN TRIM(COALESCE(census_crgo_bldgmat,    '')) <> '' THEN 1 ELSE 0 END) AS crgo_bldgmat,
    MAX(CASE WHEN TRIM(COALESCE(census_crgo_construct,  '')) <> '' THEN 1 ELSE 0 END) AS crgo_construct,
    MAX(CASE WHEN TRIM(COALESCE(census_crgo_beverages,  '')) <> '' THEN 1 ELSE 0 END) AS crgo_beverages,
    MAX(CASE WHEN TRIM(COALESCE(census_crgo_paperprod,  '')) <> '' THEN 1 ELSE 0 END) AS crgo_paperprod,
    MAX(CASE WHEN TRIM(COALESCE(census_crgo_produce,    '')) <> '' THEN 1 ELSE 0 END) AS crgo_produce,
    MAX(CASE WHEN TRIM(COALESCE(census_crgo_meat,       '')) <> '' THEN 1 ELSE 0 END) AS crgo_meat,
    MAX(CASE WHEN TRIM(COALESCE(census_crgo_drybulk,    '')) <> '' THEN 1 ELSE 0 END) AS crgo_drybulk,
    MAX(CASE WHEN TRIM(COALESCE(census_crgo_coldfood,   '')) <> '' THEN 1 ELSE 0 END) AS crgo_coldfood,
    MAX(CASE WHEN TRIM(COALESCE(census_crgo_intermodal, '')) <> '' THEN 1 ELSE 0 END) AS crgo_intermodal,
    MAX(CASE WHEN TRIM(COALESCE(census_crgo_grainfeed,  '')) <> '' THEN 1 ELSE 0 END) AS crgo_grainfeed,
    MAX(CASE WHEN TRIM(COALESCE(census_crgo_livestock,  '')) <> '' THEN 1 ELSE 0 END) AS crgo_livestock,
    MAX(CASE WHEN TRIM(COALESCE(census_crgo_motoveh,    '')) <> '' THEN 1 ELSE 0 END) AS crgo_motoveh,
    MAX(CASE WHEN TRIM(COALESCE(census_crgo_household,  '')) <> '' THEN 1 ELSE 0 END) AS crgo_household,
    MAX(CASE WHEN TRIM(COALESCE(census_crgo_passengers, '')) <> '' THEN 1 ELSE 0 END) AS crgo_passengers,
    MAX(CASE WHEN TRIM(COALESCE(census_crgo_oilfield,   '')) <> '' THEN 1 ELSE 0 END) AS crgo_oilfield,
    MAX(CASE WHEN TRIM(COALESCE(census_crgo_chem,       '')) <> '' THEN 1 ELSE 0 END) AS crgo_chem,
    MAX(CASE WHEN TRIM(COALESCE(census_crgo_logpole,    '')) <> '' THEN 1 ELSE 0 END) AS crgo_logpole,

    -- risk record (structural zero)
    COALESCE(MAX(crash_count), 0)                 AS crash_count,
    COALESCE(MAX(crash_count_last_24_months), 0)  AS crash_count_24m,
    COALESCE(MAX(crash_fatalities), 0)            AS crash_fatalities,
    COALESCE(MAX(crash_injuries), 0)              AS crash_injuries,
    COALESCE(MAX(inspection_count), 0)            AS inspection_count,
    COALESCE(MAX(inspection_oos_violations), 0)   AS inspection_oos_viol,
    COALESCE(MAX(inspections_with_oos), 0)        AS inspections_with_oos,
    COALESCE(MAX(oos_order_count), 0)             AS oos_order_count,
    COALESCE(MAX(revocation_count), 0)            AS revocation_count,

    -- SMS measures
    MAX(sms_unsafe_driv_measure) AS sms_unsafe_driv,
    MAX(sms_hos_driv_measure)    AS sms_hos_driv,
    MAX(sms_driv_fit_measure)    AS sms_driv_fit,
    MAX(sms_veh_maint_measure)   AS sms_veh_maint,
    MAX(sms_contr_subst_measure) AS sms_contr_subst,
    CASE WHEN MAX(sms_insp_total) IS NOT NULL THEN 1 ELSE 0 END AS has_sms,

    -- business profile
    ANY_VALUE(co_primary_naics)     AS primary_naics,
    MAX(co_years_in_business)       AS years_in_business,
    ANY_VALUE(co_organization_type) AS org_type,

    -- insurance history (DOT-level summaries)
    COALESCE(MAX(insurance_event_count), 0)         AS ins_event_count,
    COALESCE(MAX(insurance_distinct_insurers), 0)   AS ins_distinct_insurers,
    COALESCE(MAX(insurance_cancelled_count), 0)     AS ins_cancelled_count,
    COALESCE(MAX(insurance_replaced_count), 0)      AS ins_replaced_count,
    COALESCE(MAX(insurance_rejected_count), 0)      AS ins_rejected_count,
    COALESCE(MAX(insurance_years_with_coverage), 0) AS ins_years_coverage,
    MAX(insurance_first_year)                        AS ins_first_year,
    MAX(insurance_last_year)                         AS ins_last_year,

    -- current placements per line (raw; consolidation applied in pandas)
    ANY_VALUE(insurance_active_bipd_insurers_as_of_2026_05_26)  AS active_bipd_raw,
    ANY_VALUE(insurance_active_cargo_insurers_as_of_2026_05_26) AS active_cargo_raw,
    ANY_VALUE(insurance_wc_insurers)                             AS active_wc_raw,
    ANY_VALUE(co_wc_carrier)                                     AS wc_carrier_raw,

    -- COVERAGE DIMENSION
    MAX(TRY_CAST(coverage_amount_usd AS DOUBLE))      AS coverage_amount_usd,
    MAX(TRY_CAST(underlying_limit_usd AS DOUBLE))     AS underlying_limit_usd,
    MAX(TRY_CAST(min_required_coverage_usd AS DOUBLE)) AS min_required_coverage_usd,

    -- PRICE PROXIES / SATISFACTION SIGNALS (used as label-definers; not model features)
    -- co_wc_residual_market is a DOT-level company flag (same across all events), so it
    -- must fire regardless of recency; residual_market is event-level and windowed.
    MAX(CASE WHEN co_wc_residual_market = TRUE THEN 1
             WHEN residual_market = TRUE AND effective_date >= {_WINDOW_START_SQL} THEN 1
             ELSE 0 END) AS is_residual_market,
    MAX(CASE WHEN LOWER(COALESCE(event_status, '')) = 'active (cancellation pending)'
                   AND effective_date >= {_WINDOW_START_SQL}
              THEN 1 ELSE 0 END) AS is_cancellation_pending,
    SUM(CASE WHEN LOWER(COALESCE(event_status, '')) = 'rejected'
                  AND effective_date >= {_WINDOW_START_SQL}
             THEN 1 ELSE 0 END) AS rejected_filing_count,
    MAX(co_wc_rate_multiplier) AS wc_rate_multiplier,

    -- TIMING SIGNALS (effective / expiration dates of the current active BIPD placement)
    MAX(CASE
          WHEN LOWER(COALESCE(event_status, '')) IN ('active', 'active (cancellation pending)')
           AND LOWER(COALESCE(policy_line, '')) IN ('auto liability', 'auto')
          THEN effective_date
        END) AS current_bipd_effective_date,
    MAX(CASE
          WHEN LOWER(COALESCE(event_status, '')) IN ('active', 'active (cancellation pending)')
           AND LOWER(COALESCE(policy_line, '')) IN ('auto liability', 'auto')
          THEN expiration_date
        END) AS current_bipd_expiration_date,

    -- =================== FEATURE EXPANSION ===================

    -- Business / industry (NAICS buckets added as derived in Part 2)
    ANY_VALUE(co_lifecycle_status)      AS co_lifecycle_status,

    -- Census carrier characteristics
    ANY_VALUE(census_carship)           AS carship,
    ANY_VALUE(census_status_code)       AS census_status_code,
    ANY_VALUE(census_safety_rating)     AS safety_rating,
    MAX(census_recordable_crash_rate)   AS crash_rate_per_mil,
    MAX(census_total_intrastate_drivers) AS total_intrastate_drivers,
    MAX(census_driver_inter_total)      AS driver_inter_total,
    MAX(census_bus_units)               AS bus_units,
    MAX(CASE WHEN UPPER(TRIM(census_prior_revoke_flag)) = 'Y' THEN 1 ELSE 0 END) AS prior_revoke_flag,
    MAX(CASE WHEN census_mcsipstep IS NOT NULL THEN 1 ELSE 0 END) AS has_mcsip,

    -- Remaining commodity flags
    MAX(CASE WHEN TRIM(COALESCE(census_crgo_cargoothr, '')) <> '' THEN 1 ELSE 0 END) AS crgo_cargoothr,
    MAX(CASE WHEN TRIM(COALESCE(census_crgo_machlrg, ''))   <> '' THEN 1 ELSE 0 END) AS crgo_machlrg,
    MAX(CASE WHEN TRIM(COALESCE(census_crgo_mobilehome, '')) <> '' THEN 1 ELSE 0 END) AS crgo_mobilehome,
    MAX(CASE WHEN TRIM(COALESCE(census_crgo_coalcoke, ''))  <> '' THEN 1 ELSE 0 END) AS crgo_coalcoke,
    MAX(CASE WHEN TRIM(COALESCE(census_crgo_garbage, ''))   <> '' THEN 1 ELSE 0 END) AS crgo_garbage,
    MAX(CASE WHEN TRIM(COALESCE(census_crgo_metalsheet, '')) <> '' THEN 1 ELSE 0 END) AS crgo_metalsheet,
    MAX(CASE WHEN TRIM(COALESCE(census_crgo_drivetow, ''))  <> '' THEN 1 ELSE 0 END) AS crgo_drivetow,
    MAX(CASE WHEN TRIM(COALESCE(census_crgo_waterwell, '')) <> '' THEN 1 ELSE 0 END) AS crgo_waterwell,
    MAX(CASE WHEN TRIM(COALESCE(census_crgo_usmail, ''))    <> '' THEN 1 ELSE 0 END) AS crgo_usmail,
    MAX(CASE WHEN TRIM(COALESCE(census_crgo_utility, ''))   <> '' THEN 1 ELSE 0 END) AS crgo_utility,
    MAX(CASE WHEN TRIM(COALESCE(census_crgo_farmsupp, ''))  <> '' THEN 1 ELSE 0 END) AS crgo_farmsupp,
    MAX(CASE WHEN TRIM(COALESCE(census_crgo_liqgas, ''))    <> '' THEN 1 ELSE 0 END) AS crgo_liqgas,

    -- Risk detail
    COALESCE(MAX(crash_tow_away_count), 0)       AS crash_tow_away_count,
    COALESCE(MAX(inspection_violations), 0)      AS inspection_violations,
    COALESCE(MAX(inspection_driver_oos), 0)      AS inspection_driver_oos,
    COALESCE(MAX(inspection_vehicle_oos), 0)     AS inspection_vehicle_oos,
    COALESCE(MAX(inspection_hazmat_oos), 0)      AS inspection_hazmat_oos,
    MAX(crash_latest_date)                       AS crash_latest_date,
    MAX(inspection_latest_date)                  AS inspection_latest_date,
    MAX(oos_latest_date)                         AS oos_latest_date,
    MAX(revocation_latest_date)                  AS revocation_latest_date,

    -- SMS detail (totals + violations + alert status per BASIC)
    COALESCE(MAX(sms_insp_total), 0)             AS sms_insp_total,
    COALESCE(MAX(sms_driver_insp_total), 0)      AS sms_driver_insp_total,
    COALESCE(MAX(sms_vehicle_insp_total), 0)     AS sms_vehicle_insp_total,
    COALESCE(MAX(sms_driver_oos_insp_total), 0)  AS sms_driver_oos_insp_total,
    COALESCE(MAX(sms_vehicle_oos_insp_total), 0) AS sms_vehicle_oos_insp_total,
    COALESCE(MAX(sms_violation_count), 0)        AS sms_violation_count,
    COALESCE(MAX(sms_violation_oos_count), 0)    AS sms_violation_oos_count,
    COALESCE(MAX(sms_violation_severity_weight), 0) AS sms_violation_severity,
    COALESCE(MAX(sms_unsafe_driv_insp_w_viol), 0) AS sms_unsafe_driv_viol,
    COALESCE(MAX(sms_hos_driv_insp_w_viol), 0)    AS sms_hos_driv_viol,
    COALESCE(MAX(sms_driv_fit_insp_w_viol), 0)    AS sms_driv_fit_viol,
    COALESCE(MAX(sms_contr_subst_insp_w_viol), 0) AS sms_contr_subst_viol,
    COALESCE(MAX(sms_veh_maint_insp_w_viol), 0)   AS sms_veh_maint_viol,
    ANY_VALUE(sms_unsafe_driv_ac)                AS sms_unsafe_driv_ac,
    ANY_VALUE(sms_hos_driv_ac)                   AS sms_hos_driv_ac,
    ANY_VALUE(sms_driv_fit_ac)                   AS sms_driv_fit_ac,
    ANY_VALUE(sms_contr_subst_ac)                AS sms_contr_subst_ac,
    ANY_VALUE(sms_veh_maint_ac)                  AS sms_veh_maint_ac,

    -- Insurance recency
    MAX(insurance_first_event_date)              AS insurance_first_event_date,
    MAX(insurance_latest_event_date)             AS insurance_latest_event_date,

    -- Authority detail (DOTs with authority filings)
    ANY_VALUE(authority_bus_state_code)          AS authority_bus_state_code,
    COALESCE(MAX(authority_history_count), 0)    AS authority_history_count,
    MAX(authority_first_served_date)             AS authority_first_served_date,
    ANY_VALUE(authority_undeliverable_mail)      AS authority_undeliverable_mail,
    ANY_VALUE(authority_property_chk)            AS authority_property_chk,
    ANY_VALUE(authority_passenger_chk)           AS authority_passenger_chk,
    ANY_VALUE(authority_hhg_chk)                 AS authority_hhg_chk,
    ANY_VALUE(authority_private_auth_chk)        AS authority_private_auth_chk,
    ANY_VALUE(authority_enterprise_chk)          AS authority_enterprise_chk,
    ANY_VALUE(authority_cargo_req)               AS authority_cargo_req,
    ANY_VALUE(authority_bond_req)                AS authority_bond_req,
    ANY_VALUE(authority_common_app_pend)         AS authority_common_app_pend,
    ANY_VALUE(authority_contract_app_pend)       AS authority_contract_app_pend,
    ANY_VALUE(authority_broker_app_pend)         AS authority_broker_app_pend,
    ANY_VALUE(authority_common_rev_pend)         AS authority_common_rev_pend,
    ANY_VALUE(authority_contract_rev_pend)       AS authority_contract_rev_pend,
    ANY_VALUE(authority_broker_rev_pend)         AS authority_broker_rev_pend,

    -- Company sophistication signals
    MAX(CASE WHEN is_peo = TRUE THEN 1 ELSE 0 END) AS is_peo_flag,
    MAX(co_usdot_count)                          AS co_usdot_count,
    MAX(co_phone_count)                          AS co_phone_count,
    MAX(co_officer_count)                        AS co_officer_count,
    MAX(co_source_count)                         AS co_source_count,
    MAX(co_resolved_company_count)               AS co_resolved_company_count,
    MAX(CASE WHEN co_has_retirement_plan = TRUE THEN 1 ELSE 0 END) AS co_has_retirement_plan,
    MAX(CASE WHEN co_has_health_plan     = TRUE THEN 1 ELSE 0 END) AS co_has_health_plan,
    MAX(CASE WHEN co_primary_domain IS NOT NULL THEN 1 ELSE 0 END) AS has_domain,
    ANY_VALUE(co_match_confidence)               AS co_match_confidence,

    -- Fleet date for recency computation
    ANY_VALUE(census_mcs150_date)                AS census_mcs150_date,

    MAX(has_insurance_event) AS has_insurance_event
FROM raw
WHERE usdot IS NOT NULL
GROUP BY usdot

""")

n_dots = con.execute("SELECT COUNT(*) FROM dot_features").fetchone()[0]
n_scorable = con.execute("""
    SELECT COUNT(*) FROM dot_features
    WHERE active_bipd_raw IS NOT NULL
       OR active_cargo_raw IS NOT NULL
       OR active_wc_raw IS NOT NULL
       OR wc_carrier_raw IS NOT NULL
""").fetchone()[0]
print(f"dot_features: {n_dots:,} DOTs ({n_scorable:,} scorable with any active placement)")

_log("Part 1E complete")


In [ ]:
# ============================================================
# v3 Part 2A — Carrier lookup + consolidation report CSV
# ============================================================
_log("starting Part 2A")

print("Building carrier consolidation lookup (over LINES_IN_SCOPE)...")
distinct_insurers = con.execute(f"""
    SELECT DISTINCT UPPER(TRIM(CAST(insurer AS VARCHAR))) AS insurer_raw
    FROM raw
    WHERE insurer IS NOT NULL
      AND effective_date >= {_WINDOW_START_SQL}
      AND effective_date <= {_VINTAGE_SQL}
      AND LOWER(TRIM(CAST(policy_line AS VARCHAR))) IN
          ({', '.join("'" + L + "'" for L in LINES_IN_SCOPE)})
""").df()
distinct_insurers['carrier_group'] = distinct_insurers['insurer_raw'].map(consolidate_one)
distinct_insurers = distinct_insurers.dropna(subset=['carrier_group'])
print(f"  {len(distinct_insurers):,} distinct insurer strings → {distinct_insurers['carrier_group'].nunique():,} consolidated groups")

con.register('carrier_lookup_df', distinct_insurers)
con.execute("DROP TABLE IF EXISTS carrier_lookup")
con.execute("CREATE TABLE carrier_lookup AS SELECT * FROM carrier_lookup_df")
con.unregister('carrier_lookup_df')

# Consolidation report with per-row mapping source + tier
print("\nWriting carrier_consolidation_report.csv...")
consolidation_report = con.execute(f"""
WITH event_counts AS (
    SELECT UPPER(TRIM(CAST(r.insurer AS VARCHAR))) AS insurer_raw,
           COUNT(*) AS n_events_24m
    FROM raw r
    WHERE r.insurer IS NOT NULL
      AND r.effective_date >= {_WINDOW_START_SQL}
      AND r.effective_date <= {_VINTAGE_SQL}
      AND LOWER(TRIM(CAST(r.policy_line AS VARCHAR))) IN
          ({', '.join("'" + L + "'" for L in LINES_IN_SCOPE)})
    GROUP BY 1
)
SELECT cl.carrier_group, cl.insurer_raw,
       COALESCE(ec.n_events_24m, 0) AS n_events_24m
FROM carrier_lookup cl
LEFT JOIN event_counts ec USING (insurer_raw)
ORDER BY cl.carrier_group, n_events_24m DESC, cl.insurer_raw
""").df()

def _mapping_source(ins_raw, grp):
    if DATA_PROVIDER_MAP.get(ins_raw) == grp:
        return ('provider', None)
    rule_grp, rule_tier = _apply_rules_only(ins_raw)
    if rule_grp == grp:
        return (f'rule_T{rule_tier}', rule_tier)
    if grp == ins_raw:
        return ('raw_fallback', None)
    return ('other', None)

_sources = consolidation_report.apply(
    lambda r: _mapping_source(r['insurer_raw'], r['carrier_group']), axis=1
)
consolidation_report['mapping_source'] = _sources.str[0]
consolidation_report['rule_tier']      = _sources.str[1]

grp_total = consolidation_report.groupby('carrier_group')['n_events_24m'].transform('sum')
consolidation_report['group_total_events']  = grp_total
consolidation_report['pct_of_group'] = (
    100.0 * consolidation_report['n_events_24m'] / consolidation_report['group_total_events'].replace(0, np.nan)
).round(2)
consolidation_report['n_entities_in_group'] = consolidation_report.groupby('carrier_group')['insurer_raw'].transform('nunique')

consolidation_report.to_csv('carrier_consolidation_report.csv', index=False)
print(f"  Wrote carrier_consolidation_report.csv ({len(consolidation_report):,} rows, "
      f"{consolidation_report['carrier_group'].nunique():,} groups)")

grp_summary = (consolidation_report.drop_duplicates('carrier_group')
               [['carrier_group','n_entities_in_group','group_total_events']]
               .sort_values('group_total_events', ascending=False).head(20))
print("\n  Top 20 groups by event volume:")
print(grp_summary.to_string(index=False))

_log("Part 2A complete")


In [ ]:
# ============================================================
# v3 Part 2B - recent_events (24mo window anchored at data vintage, state normalised)
# ============================================================
_log("starting Part 2B - recent_events")

con.execute("DROP TABLE IF EXISTS recent_events")
con.execute(f"""
CREATE TABLE recent_events AS
SELECT r.usdot,
       cl.insurer_raw,
       cl.carrier_group,
       CASE
         WHEN LOWER(TRIM(CAST(r.policy_line AS VARCHAR))) IN ('auto liability','auto') THEN 'bipd'
         WHEN LOWER(TRIM(CAST(r.policy_line AS VARCHAR))) = 'broker surety bond'       THEN 'surety_bond'
         ELSE LOWER(TRIM(CAST(r.policy_line AS VARCHAR)))
       END AS line,
       LOWER(TRIM(CAST(r.event_status AS VARCHAR))) AS event_status,
       LOWER(TRIM(CAST(r.end_reason   AS VARCHAR))) AS end_reason,
       r.effective_date,
       -- Validate cancellation_or_end_date: null only year-9xxx entry errors.
       -- The year-2100 ceiling is a sanity cap; multi-decade surety bonds and
       -- other long-term contracts are kept.
       CASE WHEN r.cancellation_or_end_date BETWEEN DATE '1990-01-01'
                                                AND DATE '2100-01-01'
            THEN r.cancellation_or_end_date END AS cancellation_or_end_date,
       DATE_DIFF('day', r.effective_date,
           CASE WHEN r.cancellation_or_end_date BETWEEN DATE '1990-01-01'
                                                    AND DATE '2100-01-01'
                THEN r.cancellation_or_end_date END) AS days_in_force,
       UPPER(TRIM(COALESCE(r.census_phy_state, r.state))) AS state,
       CASE WHEN r.residual_market = TRUE OR r.co_wc_residual_market = TRUE
            THEN 1 ELSE 0 END AS is_residual_market_event
FROM raw r
JOIN carrier_lookup cl
  ON UPPER(TRIM(CAST(r.insurer AS VARCHAR))) = cl.insurer_raw
WHERE r.insurer IS NOT NULL
  AND r.effective_date >= {_WINDOW_START_SQL}
  AND r.effective_date <= {_VINTAGE_SQL}
  AND LOWER(TRIM(CAST(r.policy_line AS VARCHAR))) IN
      ({', '.join("'" + L + "'" for L in LINES_IN_SCOPE)})
""")

n_recent = con.execute("SELECT COUNT(*) FROM recent_events").fetchone()[0]
per_line = con.execute("SELECT line, COUNT(*) AS n FROM recent_events GROUP BY line ORDER BY n DESC").df()
print(f"  {n_recent:,} recent events in [{_WINDOW_START.date()}, {_DATA_VINTAGE.date()}]")
print(per_line.to_string(index=False))

_log_mem("after recent_events")
_log("Part 2B complete")


In [ ]:
# ============================================================
# v3 Part 2C - Service composite + (carrier, state, line) aggregates + momentum
# ============================================================
# Adds carr_placements_6m, carr_placements_6to18m, carr_state_momentum
# to capture carrier state entry/exit trajectory
_log("starting Part 2C")

# Service composite
print("Computing service composite per carrier...")
con.execute("DROP TABLE IF EXISTS service_composite")
con.execute("""
CREATE TABLE service_composite AS
WITH stats AS (
    SELECT carrier_group,
           COUNT(*) AS svc_n_events,
           AVG(CASE WHEN days_in_force < 90 AND event_status = 'cancelled' THEN 1.0 ELSE 0.0 END)
             AS svc_short_tenure_cancel_rate,
           AVG(CASE WHEN event_status = 'rejected' THEN 1.0 ELSE 0.0 END) AS svc_rejection_rate,
           AVG(CASE WHEN days_in_force >= 0 THEN days_in_force END) AS svc_mean_tenure_days
    FROM recent_events GROUP BY carrier_group
),
vol_exit AS (
    SELECT carrier_group,
           AVG(CASE WHEN event_status = 'replaced' THEN 1.0 ELSE 0.0 END) AS svc_voluntary_exit_ratio
    FROM recent_events
    WHERE event_status IN ('cancelled','replaced')
    GROUP BY carrier_group
)
SELECT s.*, COALESCE(v.svc_voluntary_exit_ratio, 0.5) AS svc_voluntary_exit_ratio
FROM stats s LEFT JOIN vol_exit v USING (carrier_group)
""")

print("\nComputing (carrier, state, line) aggregates with momentum features...")
min_thresh_sql = ' '.join(f"WHEN line = '{line}' THEN {thresh}" for line, thresh in MIN_CARRIER_PLACEMENTS.items())

con.execute("DROP TABLE IF EXISTS carrier_state_agg")
con.execute(f"""
CREATE TABLE carrier_state_agg AS
WITH enriched AS (
    SELECT re.carrier_group, re.state, re.line, re.usdot, re.effective_date,
           re.is_residual_market_event,
           df.power_units, df.crash_count, df.inspection_oos_viol,
           df.revocation_count, df.oos_order_count,
           df.crgo_genfreight, df.crgo_bldgmat, df.crgo_construct,
           df.crgo_livestock, df.crgo_oilfield, df.crgo_passengers,
           df.crgo_chem, df.crgo_meat, df.crgo_intermodal, df.crgo_motoveh,
           CASE WHEN UPPER(TRIM(df.hazmat_ind)) IN ('Y','X') THEN 1.0 ELSE 0.0 END AS hazmat_bin,
           df.inter_beyond_100, df.intra_within_100
    FROM recent_events re
    JOIN dot_features df USING (usdot)
),
agg AS (
    SELECT carrier_group, state, line,
           COUNT(DISTINCT usdot) AS carr_n_placements,
           COUNT(*) AS carr_n_events,
           -- Means (overwritten by LOO at train/score time; see apply_loo_adjustment)
           AVG(power_units) AS carr_mean_power,
           MEDIAN(power_units) AS carr_median_power,
           QUANTILE_CONT(power_units, 0.9) AS carr_p90_power,
           AVG(crash_count) AS carr_mean_crash,
           AVG(inspection_oos_viol) AS carr_mean_insp_oos,
           AVG(revocation_count) AS carr_mean_revoc,
           AVG(oos_order_count) AS carr_mean_oos,
           AVG(hazmat_bin) AS carr_pct_hazmat,
           AVG(crgo_genfreight) AS carr_pct_genfreight,
           AVG(crgo_bldgmat) AS carr_pct_bldgmat,
           AVG(crgo_construct) AS carr_pct_construct,
           AVG(crgo_livestock) AS carr_pct_livestock,
           AVG(crgo_oilfield) AS carr_pct_oilfield,
           AVG(crgo_passengers) AS carr_pct_passengers,
           AVG(crgo_chem) AS carr_pct_chem,
           AVG(crgo_meat) AS carr_pct_meat,
           AVG(crgo_intermodal) AS carr_pct_intermodal,
           AVG(crgo_motoveh) AS carr_pct_motoveh,
           AVG(inter_beyond_100) AS carr_mean_inter_long,
           AVG(intra_within_100) AS carr_mean_intra_short,
           -- Share of residual-market events (for filtering recommendations)
           AVG(CAST(is_residual_market_event AS DOUBLE)) AS carr_pct_residual_market,
           -- Sums (for LOO adjustment at train/score time)
           SUM(power_units)        AS carr_power_sum,
           SUM(crash_count)        AS carr_crash_sum,
           SUM(inspection_oos_viol) AS carr_insp_oos_sum,
           SUM(revocation_count)   AS carr_revoc_sum,
           SUM(oos_order_count)    AS carr_oos_sum,
           SUM(hazmat_bin)         AS carr_hazmat_sum,
           SUM(crgo_genfreight)    AS carr_genfreight_sum,
           SUM(crgo_bldgmat)       AS carr_bldgmat_sum,
           SUM(crgo_construct)     AS carr_construct_sum,
           SUM(crgo_livestock)     AS carr_livestock_sum,
           SUM(crgo_oilfield)      AS carr_oilfield_sum,
           SUM(crgo_passengers)    AS carr_passengers_sum,
           SUM(crgo_chem)          AS carr_chem_sum,
           SUM(crgo_meat)          AS carr_meat_sum,
           SUM(crgo_intermodal)    AS carr_intermodal_sum,
           SUM(crgo_motoveh)       AS carr_motoveh_sum,
           SUM(inter_beyond_100)   AS carr_inter_long_sum,
           SUM(intra_within_100)   AS carr_intra_short_sum,
           COUNT(DISTINCT CASE WHEN effective_date >= CAST({_VINTAGE_SQL} AS DATE) - INTERVAL '6 months'
                               THEN usdot END) AS carr_placements_6m,
           COUNT(DISTINCT CASE WHEN effective_date >= CAST({_VINTAGE_SQL} AS DATE) - INTERVAL '18 months'
                                AND effective_date <  CAST({_VINTAGE_SQL} AS DATE) - INTERVAL '6 months'
                               THEN usdot END) AS carr_placements_6to18m
    FROM enriched
    GROUP BY carrier_group, state, line
)
SELECT a.*,
       sc.svc_short_tenure_cancel_rate, sc.svc_rejection_rate,
       sc.svc_voluntary_exit_ratio, sc.svc_mean_tenure_days,
       CAST(a.carr_placements_6m AS DOUBLE) / NULLIF(a.carr_placements_6to18m / 2.0 + 1, 0)
         AS carr_state_momentum
FROM agg a
LEFT JOIN service_composite sc USING (carrier_group)
WHERE a.carr_n_placements >= CASE {min_thresh_sql} ELSE 20 END
""")

n_cells = con.execute("SELECT COUNT(*) FROM carrier_state_agg").fetchone()[0]
print(f"  {n_cells:,} (carrier, state, line) cells after per-line min-N")

carrier_state_agg = con.execute("SELECT * FROM carrier_state_agg").df()
for c in carrier_state_agg.columns:
    if pd.api.types.is_integer_dtype(carrier_state_agg[c]):
        carrier_state_agg[c] = pd.to_numeric(carrier_state_agg[c], downcast='integer')
    elif pd.api.types.is_float_dtype(carrier_state_agg[c]):
        carrier_state_agg[c] = pd.to_numeric(carrier_state_agg[c], downcast='float')
print(f"  cells per line: {carrier_state_agg.groupby('line').size().to_dict()}")

# ------------------------------------------------------------
# dot_cell_contrib — per (usdot, carrier, state, line), the DOT's event
# count in that cell. Used for leave-one-out adjustment of carrier aggregates.
# ------------------------------------------------------------
con.execute(f"""
CREATE OR REPLACE TABLE dot_cell_contrib AS
SELECT re.usdot, re.carrier_group, re.state, re.line,
       COUNT(*) AS dot_n_events_in_cell,
       MAX(CASE WHEN re.effective_date >= CAST({_VINTAGE_SQL} AS DATE) - INTERVAL '6 months'
                THEN 1 ELSE 0 END) AS dot_in_6m,
       MAX(CASE WHEN re.effective_date >= CAST({_VINTAGE_SQL} AS DATE) - INTERVAL '18 months'
                 AND re.effective_date <  CAST({_VINTAGE_SQL} AS DATE) - INTERVAL '6 months'
                THEN 1 ELSE 0 END) AS dot_in_6to18m
FROM recent_events re
GROUP BY re.usdot, re.carrier_group, re.state, re.line
""")

dot_cell_contrib = con.execute("SELECT * FROM dot_cell_contrib").df()
for c in ['dot_n_events_in_cell','dot_in_6m','dot_in_6to18m']:
    dot_cell_contrib[c] = pd.to_numeric(dot_cell_contrib[c], downcast='integer')
print(f"  dot_cell_contrib: {len(dot_cell_contrib):,} DOT-cell pairs (for LOO)")

_log("Part 2C complete")


In [ ]:
# ============================================================
# v3 Part 2D - Sub-entity centroids + Option B feature weights
# ============================================================
_log("starting Part 2D")

con.execute("DROP TABLE IF EXISTS sub_entity_centroids")
con.execute(f"""
CREATE TABLE sub_entity_centroids AS
WITH enriched AS (
    SELECT re.carrier_group, re.state, re.line, re.insurer_raw, re.usdot,
           LN(1.0 + COALESCE(df.power_units, 0)) AS power_units_log,
           COALESCE(df.crash_count, 0)           AS crash_count,
           COALESCE(df.inspection_oos_viol, 0)   AS inspection_oos_viol,
           COALESCE(df.crgo_genfreight, 0)       AS crgo_genfreight,
           COALESCE(df.crgo_bldgmat, 0)          AS crgo_bldgmat,
           COALESCE(df.crgo_livestock, 0)        AS crgo_livestock,
           COALESCE(df.crgo_oilfield, 0)         AS crgo_oilfield,
           COALESCE(df.crgo_intermodal, 0)       AS crgo_intermodal,
           COALESCE(df.crgo_chem, 0)             AS crgo_chem,
           COALESCE(df.crgo_passengers, 0)       AS crgo_passengers
    FROM recent_events re
    JOIN dot_features df USING (usdot)
)
SELECT carrier_group, state, line, insurer_raw,
       COUNT(DISTINCT usdot) AS sub_n_placements,
       COUNT(*)              AS sub_n_events,
       -- Means (LOO-adjusted at distance time; see best_subentity_for_dot)
       AVG(power_units_log)      AS sub_power_units_log,
       AVG(crash_count)          AS sub_crash_count,
       AVG(inspection_oos_viol)  AS sub_inspection_oos_viol,
       AVG(crgo_genfreight)      AS sub_crgo_genfreight,
       AVG(crgo_bldgmat)         AS sub_crgo_bldgmat,
       AVG(crgo_livestock)       AS sub_crgo_livestock,
       AVG(crgo_oilfield)        AS sub_crgo_oilfield,
       AVG(crgo_intermodal)      AS sub_crgo_intermodal,
       AVG(crgo_chem)            AS sub_crgo_chem,
       AVG(crgo_passengers)      AS sub_crgo_passengers,
       -- Sums (for LOO adjustment in Option B)
       SUM(power_units_log)      AS sub_power_units_log_sum,
       SUM(crash_count)          AS sub_crash_count_sum,
       SUM(inspection_oos_viol)  AS sub_inspection_oos_viol_sum,
       SUM(crgo_genfreight)      AS sub_crgo_genfreight_sum,
       SUM(crgo_bldgmat)         AS sub_crgo_bldgmat_sum,
       SUM(crgo_livestock)       AS sub_crgo_livestock_sum,
       SUM(crgo_oilfield)        AS sub_crgo_oilfield_sum,
       SUM(crgo_intermodal)      AS sub_crgo_intermodal_sum,
       SUM(crgo_chem)            AS sub_crgo_chem_sum,
       SUM(crgo_passengers)      AS sub_crgo_passengers_sum
FROM enriched
GROUP BY carrier_group, state, line, insurer_raw
HAVING COUNT(DISTINCT usdot) >= {MIN_SUB_ENTITY_PLACEMENTS}
""")

sub_entity_centroids = con.execute("SELECT * FROM sub_entity_centroids").df()
for c in sub_entity_centroids.columns:
    if pd.api.types.is_float_dtype(sub_entity_centroids[c]):
        sub_entity_centroids[c] = sub_entity_centroids[c].astype('float32')
print(f"  {len(sub_entity_centroids):,} sub-entity centroids")

# Within-cell variance -> feature weights for Option B distance
print("\nComputing within-cell feature variances for distance weighting...")
dist_feat_cols = [f'sub_{f}' for f in APPETITE_DIST_FEATURES]
cell_sizes = sub_entity_centroids.groupby(['carrier_group','state','line']).size()
multi_cells = cell_sizes[cell_sizes >= 2].index
if len(multi_cells) > 0:
    v = (sub_entity_centroids
         .merge(pd.DataFrame(multi_cells.tolist(), columns=['carrier_group','state','line']),
                on=['carrier_group','state','line'], how='inner')
         .groupby(['carrier_group','state','line'])[dist_feat_cols]
         .var().fillna(0))
    feat_var = v.mean(axis=0)
    FEATURE_WEIGHTS = (feat_var / feat_var.sum()).to_dict() if feat_var.sum() > 0 else {}
else:
    FEATURE_WEIGHTS = {c: 1.0 / len(dist_feat_cols) for c in dist_feat_cols}

print("  Appetite-distance feature weights (sum to 1):")
for k, v in sorted(FEATURE_WEIGHTS.items(), key=lambda x: -x[1]):
    print(f"    {k:<30s}  {v:.4f}")

# ------------------------------------------------------------
# dot_subent_contrib — per (usdot, carrier, state, line, insurer_raw),
# the DOT's event count in that sub-entity. Used for LOO in Option B distance.
# ------------------------------------------------------------
con.execute("""
CREATE OR REPLACE TABLE dot_subent_contrib AS
SELECT re.usdot, re.carrier_group, re.state, re.line, re.insurer_raw,
       COUNT(*) AS dot_n_events_in_subent
FROM recent_events re
GROUP BY re.usdot, re.carrier_group, re.state, re.line, re.insurer_raw
""")
dot_subent_contrib = con.execute("SELECT * FROM dot_subent_contrib").df()
dot_subent_contrib['dot_n_events_in_subent'] = pd.to_numeric(
    dot_subent_contrib['dot_n_events_in_subent'], downcast='integer')
print(f"  dot_subent_contrib: {len(dot_subent_contrib):,} DOT-subentity pairs")

_log("Part 2D complete")


In [ ]:
# ============================================================
# v3 Part 2E - Load scorable DOTs into pandas + derived features
# ============================================================
# Includes: timing, coverage ratio, log fleet, NAICS bucketing, days_since_*,
#           NEW ratio features, NEW rare-event flags, fleet_band
_log("starting Part 2E")

# ------------------------------------------------------------
# Outlier-nulling bounds (data-quality fix for raw CSV).
# Past-only columns capped at today — a 2028 crash_latest_date is a
# data error. Future-allowed columns capped at year 2100 — hard
# "obviously-a-data-error" ceiling that catches 9999-01-01 entries
# but keeps multi-decade surety bonds / long-term contracts untouched.
# ------------------------------------------------------------
_VALID_DATE_MIN      = pd.Timestamp('1990-01-01')
_VALID_DATE_MAX_PAST = pd.Timestamp.today().normalize()
_VALID_DATE_MAX_FUT  = pd.Timestamp('2100-01-01')

# (col, max_bound) — most dates are past-only; a few can be future-dated.
_DATE_COLS_PAST = [
    'crash_latest_date', 'inspection_latest_date', 'oos_latest_date',
    'revocation_latest_date', 'insurance_first_event_date',
    'insurance_latest_event_date', 'authority_first_served_date',
    'census_mcs150_date', 'current_bipd_effective_date',
]
_DATE_COLS_FUT = [
    'current_bipd_expiration_date',  # policies legitimately expire 1-2 years ahead
]
_NUMERIC_BOUNDS = {
    # Fleet / drivers
    'power_units': (0, 100_000), 'total_drivers': (0, 100_000),
    'total_cdl': (0, 100_000), 'truck_units': (0, 100_000),
    'bus_units': (0, 100_000), 'total_intrastate_drivers': (0, 100_000),
    'driver_inter_total': (0, 100_000), 'mcs150_mileage': (0, 1_000_000_000),
    # Crashes / inspections / enforcement
    'crash_count': (0, 10_000), 'crash_count_24m': (0, 10_000),
    'crash_fatalities': (0, 1_000), 'crash_injuries': (0, 10_000),
    'crash_tow_away_count': (0, 10_000),
    'inspection_count': (0, 1_000_000), 'inspection_violations': (0, 1_000_000),
    'inspection_oos_viol': (0, 100_000), 'inspections_with_oos': (0, 100_000),
    'inspection_driver_oos': (0, 100_000), 'inspection_vehicle_oos': (0, 100_000),
    'inspection_hazmat_oos': (0, 10_000),
    'oos_order_count': (0, 10_000), 'revocation_count': (0, 1_000),
    # Insurance history
    'ins_event_count': (0, 10_000), 'ins_distinct_insurers': (0, 1_000),
    'ins_cancelled_count': (0, 10_000), 'ins_replaced_count': (0, 10_000),
    'ins_rejected_count': (0, 10_000), 'ins_years_coverage': (0, 100),
    'ins_first_year': (1960, 2035), 'ins_last_year': (1960, 2035),
    # Coverage / price
    'coverage_amount_usd': (0, 100_000_000_000),
    'underlying_limit_usd': (0, 100_000_000_000),
    'min_required_coverage_usd': (0, 100_000_000_000),
    'authority_min_cov_amount': (0, 100_000_000_000),
    'authority_history_count': (0, 10_000),
    # Business profile
    'years_in_business': (0, 150),
    'co_usdot_count': (0, 100_000), 'co_phone_count': (0, 100_000),
    'co_officer_count': (0, 10_000), 'co_source_count': (0, 10_000),
    'co_resolved_company_count': (0, 10_000),
    'crash_rate_per_mil': (0, 10_000), 'wc_rate_multiplier': (0, 100),
}

print("Loading scorable DOTs into pandas...")
dot_df = con.execute("""
    WITH event_active AS (
        SELECT DISTINCT usdot FROM recent_events
        WHERE line IN ('surety_bond','surety','gl')
          AND event_status IN ('active','active (cancellation pending)')
    )
    SELECT * FROM dot_features df
    WHERE df.active_bipd_raw IS NOT NULL
       OR df.active_cargo_raw IS NOT NULL
       OR df.active_wc_raw IS NOT NULL
       OR df.wc_carrier_raw IS NOT NULL
       OR df.usdot IN (SELECT usdot FROM event_active)
""").df()
print(f"  {len(dot_df):,} scorable DOTs")
_log_mem("after dot_df load")

# Downcast dtypes
for c in dot_df.columns:
    if pd.api.types.is_integer_dtype(dot_df[c]):
        dot_df[c] = pd.to_numeric(dot_df[c], downcast='integer')
    elif pd.api.types.is_float_dtype(dot_df[c]):
        dot_df[c] = pd.to_numeric(dot_df[c], downcast='float')
_log_mem("after dot_df downcast")

# ------------------------------------------------------------
# Null outlier dates (raw has entries like 9001-01-01 and 2028-xx).
# Set to NaT so downstream days_since_* becomes NaN rather than
# millions-of-days (or large-negative for "future" dates).
# ------------------------------------------------------------
print("Sanitizing outlier dates in dot_df...")
_date_oob_summary = []
for col, cap in ([(c, _VALID_DATE_MAX_PAST) for c in _DATE_COLS_PAST]
                 + [(c, _VALID_DATE_MAX_FUT) for c in _DATE_COLS_FUT]):
    if col not in dot_df.columns: continue
    s = pd.to_datetime(dot_df[col], errors='coerce')
    oob = (s < _VALID_DATE_MIN) | (s > cap)
    n_oob = int(oob.sum())
    if n_oob:
        _date_oob_summary.append((col, n_oob, s[oob].min(), s[oob].max(), cap))
    dot_df[col] = s.mask(oob)
if _date_oob_summary:
    print(f"  Nulled {sum(x[1] for x in _date_oob_summary):,} out-of-range dates across "
          f"{len(_date_oob_summary)} columns:")
    for col, n, lo, hi, cap in _date_oob_summary:
        print(f"    {col:38s} {n:>7,} values  (observed {lo} .. {hi}; cap {cap.date()})")
else:
    print("  No date outliers detected.")

# ------------------------------------------------------------
# Null outlier numerics (raw has entries like power_units in the millions).
# Clipping would pretend a 1e9 is a legit 100k and poison the LOO sums.
# Nulling is honest ("we don't know"); DuckDB SUM/AVG skip NULL, so LOO
# aggregates stay clean, and LightGBM / HGBT handle NaN natively.
# ------------------------------------------------------------
print("Nulling outlier numerics in dot_df...")
_num_oob_summary = []
for col, (lo, hi) in _NUMERIC_BOUNDS.items():
    if col not in dot_df.columns: continue
    s = pd.to_numeric(dot_df[col], errors='coerce')
    oob = (s < lo) | (s > hi)
    n_oob = int(oob.sum())
    if n_oob:
        _num_oob_summary.append((col, n_oob, float(s[oob].min()), float(s[oob].max())))
    dot_df[col] = s.mask(oob)
if _num_oob_summary:
    print(f"  Nulled {sum(x[1] for x in _num_oob_summary):,} out-of-range values across "
          f"{len(_num_oob_summary)} columns:")
    for col, n, lo_, hi_ in _num_oob_summary:
        print(f"    {col:38s} {n:>7,} values  (observed range {lo_:.2f} .. {hi_:.2f})")
else:
    print("  No numeric outliers detected.")

# ------------------------------------------------------------
# Timing features — anchored at data vintage (not today!)
# ------------------------------------------------------------
today = _DATA_VINTAGE
dot_df['current_bipd_effective_date']  = pd.to_datetime(dot_df['current_bipd_effective_date'], errors='coerce')
dot_df['current_bipd_expiration_date'] = pd.to_datetime(dot_df['current_bipd_expiration_date'], errors='coerce')
dot_df['days_since_bipd_effective']  = ((today - dot_df['current_bipd_effective_date']).dt.days).astype('float32')
dot_df['days_until_bipd_expiration'] = ((dot_df['current_bipd_expiration_date'] - today).dt.days).astype('float32')
dot_df['years_with_current_carrier'] = (dot_df['days_since_bipd_effective'] / 365.25).astype('float32')

# Coverage-fit derived
dot_df['coverage_vs_minimum_ratio']  = (dot_df['coverage_amount_usd']
                                        / dot_df['authority_min_cov_amount'].replace(0, np.nan)).astype('float32')
dot_df['has_layered_coverage']       = dot_df['underlying_limit_usd'].notna().astype('int8')
dot_df['power_units_log']            = np.log1p(dot_df['power_units'].fillna(0)).astype('float32')
dot_df['hazmat_bin']                 = (dot_df['hazmat_ind'].astype(str).str.strip().str.upper()
                                        .isin(['Y','X'])).astype('int8')

# ------------------------------------------------------------
# NAICS bucketing
# ------------------------------------------------------------
dot_df['naics_sector']    = dot_df['primary_naics'].astype(str).str[:3].where(dot_df['primary_naics'].notna(), 'UNK')
dot_df['naics_subsector'] = dot_df['primary_naics'].astype(str).str[:4].where(dot_df['primary_naics'].notna(), 'UNK')

# ------------------------------------------------------------
# Days-since-* recency features (anchored at data vintage)
# ------------------------------------------------------------
def _days_since(col):
    s = pd.to_datetime(dot_df[col], errors='coerce')
    return ((today - s).dt.days).astype('float32')

for src_col, feat_col in [
    ('crash_latest_date', 'days_since_last_crash'),
    ('inspection_latest_date', 'days_since_last_inspection'),
    ('oos_latest_date', 'days_since_last_oos'),
    ('revocation_latest_date', 'days_since_last_revocation'),
    ('insurance_first_event_date', 'days_since_insurance_first'),
    ('insurance_latest_event_date', 'days_since_insurance_latest'),
    ('authority_first_served_date', 'days_since_authority_served'),
]:
    if src_col in dot_df.columns:
        dot_df[feat_col] = _days_since(src_col)

if 'census_mcs150_date' in dot_df.columns:
    _mcs150 = pd.to_datetime(dot_df['census_mcs150_date'], errors='coerce')
    dot_df['days_since_mcs150_update'] = ((today - _mcs150).dt.days).astype('float32')

# ------------------------------------------------------------
# NEW: Ratio features (rates normalised by fleet / inspections / history)
# ------------------------------------------------------------
dot_df['crash_rate_per_pu']        = (dot_df['crash_count']         / (dot_df['power_units'] + 1)).astype('float32')
dot_df['oos_rate_per_pu']          = (dot_df['oos_order_count']     / (dot_df['power_units'] + 1)).astype('float32')
dot_df['violation_rate']           = (dot_df['inspection_violations']/ (dot_df['inspection_count'] + 1)).astype('float32')
dot_df['oos_violation_rate']       = (dot_df['inspection_oos_viol'] / (dot_df['inspection_count'] + 1)).astype('float32')
dot_df['driver_oos_rate']          = (dot_df['inspection_driver_oos']/ (dot_df['inspection_count'] + 1)).astype('float32')
dot_df['vehicle_oos_rate']         = (dot_df['inspection_vehicle_oos']/ (dot_df['inspection_count'] + 1)).astype('float32')
dot_df['hazmat_oos_rate']          = (dot_df['inspection_hazmat_oos']/ (dot_df['inspection_count'] + 1)).astype('float32')
dot_df['insurance_cancel_rate']    = (dot_df['ins_cancelled_count'] / (dot_df['ins_event_count'] + 1)).astype('float32')
dot_df['insurance_rejection_rate'] = (dot_df['ins_rejected_count']  / (dot_df['ins_event_count'] + 1)).astype('float32')
dot_df['carrier_switch_rate']      = (dot_df['ins_distinct_insurers']/ (dot_df['ins_years_coverage'] + 1)).astype('float32')
dot_df['driver_per_pu']            = (dot_df['total_drivers']       / (dot_df['power_units'] + 1)).astype('float32')

# Age of insurance history (clipped to >= 0)
dot_df['insurance_history_years']  = ((dot_df['ins_last_year'] - dot_df['ins_first_year']).fillna(0).clip(lower=0)).astype('float32')

# ------------------------------------------------------------
# NEW: Rare-event flags
# ------------------------------------------------------------
dot_df['had_any_fatal_crash']   = (dot_df['crash_fatalities'].fillna(0) > 0).astype('int8')
dot_df['had_any_revocation']    = (dot_df['revocation_count'].fillna(0) > 0).astype('int8')
dot_df['had_any_oos_order']     = (dot_df['oos_order_count'].fillna(0) > 0).astype('int8')
dot_df['had_any_rejection']     = (dot_df['ins_rejected_count'].fillna(0) > 0).astype('int8')

# ------------------------------------------------------------
# Fleet band (moved from Part 4 to here)
# ------------------------------------------------------------
def _fleet_band(pu):
    if pu is None or pd.isna(pu): return 'UNK'
    pu = float(pu)
    if pu <= 1:   return 'B1'
    if pu <= 5:   return 'B2'
    if pu <= 20:  return 'B3'
    if pu <= 100: return 'B4'
    return 'B5'

dot_df['fleet_band'] = dot_df['power_units'].map(_fleet_band)

print(f"\nDerived features: {dot_df.shape[1]} columns in dot_df")
_log_mem("end of Part 2E")
_log("Part 2E complete")


In [ ]:
# ============================================================
# v3 Part 3A - Feature definitions + CAT_MAPS + classifier factory
# ============================================================
_log("starting Part 3A")

DOT_NUM_FEATURES = [
    # Fleet shape
    'power_units', 'power_units_log', 'total_drivers', 'total_cdl', 'truck_units',
    'mcs150_mileage', 'bus_units',
    # Driver composition
    'total_intrastate_drivers', 'driver_inter_total',
    # Radius
    'inter_beyond_100', 'inter_within_100', 'intra_beyond_100', 'intra_within_100',
    # Risk record
    'crash_count', 'crash_count_24m', 'crash_fatalities', 'crash_injuries', 'crash_tow_away_count',
    'crash_rate_per_mil',
    'inspection_count', 'inspection_violations',
    'inspection_oos_viol', 'inspections_with_oos',
    'inspection_driver_oos', 'inspection_vehicle_oos', 'inspection_hazmat_oos',
    'oos_order_count', 'revocation_count', 'prior_revoke_flag',
    # SMS detail
    'sms_insp_total', 'sms_driver_insp_total', 'sms_vehicle_insp_total',
    'sms_driver_oos_insp_total', 'sms_vehicle_oos_insp_total',
    'sms_violation_count', 'sms_violation_oos_count', 'sms_violation_severity',
    'sms_unsafe_driv', 'sms_hos_driv', 'sms_driv_fit', 'sms_veh_maint', 'sms_contr_subst',
    'sms_unsafe_driv_viol', 'sms_hos_driv_viol', 'sms_driv_fit_viol',
    'sms_contr_subst_viol', 'sms_veh_maint_viol',
    'has_sms',
    # Authority
    'has_authority', 'authority_history_count',
    # Business
    'years_in_business', 'co_usdot_count', 'co_phone_count', 'co_officer_count',
    'co_source_count', 'co_resolved_company_count',
    'co_has_retirement_plan', 'co_has_health_plan', 'has_domain',
    'is_peo_flag', 'has_mcsip',
    # Insurance history
    'ins_event_count', 'ins_distinct_insurers',
    'ins_cancelled_count', 'ins_replaced_count', 'ins_rejected_count',
    'ins_years_coverage',
    # Commodity flags (30 total)
    'crgo_genfreight', 'crgo_bldgmat', 'crgo_construct', 'crgo_beverages',
    'crgo_paperprod', 'crgo_produce', 'crgo_meat', 'crgo_drybulk', 'crgo_coldfood',
    'crgo_intermodal', 'crgo_grainfeed', 'crgo_livestock', 'crgo_motoveh',
    'crgo_household', 'crgo_passengers', 'crgo_oilfield', 'crgo_chem', 'crgo_logpole',
    'crgo_cargoothr', 'crgo_machlrg', 'crgo_mobilehome', 'crgo_coalcoke',
    'crgo_garbage', 'crgo_metalsheet', 'crgo_drivetow', 'crgo_waterwell',
    'crgo_usmail', 'crgo_utility', 'crgo_farmsupp', 'crgo_liqgas',
    # Coverage / price proxies
    'coverage_amount_usd', 'underlying_limit_usd', 'min_required_coverage_usd',
    'authority_min_cov_amount', 'coverage_vs_minimum_ratio', 'has_layered_coverage',
    'rejected_filing_count',
    'wc_rate_multiplier',
    # NOTE: is_residual_market and is_cancellation_pending removed from features —
    # they are used in Part 3C to define positive/negative labels; keeping them as
    # features would cause direct label leakage.
    # Timing / recency
    'days_since_bipd_effective', 'days_until_bipd_expiration', 'years_with_current_carrier',
    'days_since_last_crash', 'days_since_last_inspection',
    'days_since_last_oos', 'days_since_last_revocation',
    'days_since_insurance_first', 'days_since_insurance_latest',
    'days_since_authority_served', 'days_since_mcs150_update',
    # Insurance event flag
    'has_insurance_event',
    # NEW: ratio features
    'crash_rate_per_pu', 'oos_rate_per_pu', 'violation_rate', 'oos_violation_rate',
    'driver_oos_rate', 'vehicle_oos_rate', 'hazmat_oos_rate',
    'insurance_cancel_rate', 'insurance_rejection_rate', 'carrier_switch_rate',
    'driver_per_pu', 'insurance_history_years',
    # NEW: rare-event flags
    'had_any_fatal_crash', 'had_any_revocation', 'had_any_oos_order', 'had_any_rejection',
]

DOT_CAT_FEATURES = [
    'state', 'phy_state', 'authority_bus_state_code',
    'naics_sector', 'naics_subsector', 'co_lifecycle_status', 'org_type',
    'fleetsize_band', 'carrier_operation', 'classdef', 'hazmat_ind',
    'carship', 'census_status_code', 'safety_rating', 'fleet_band',
    'authority_common_stat', 'authority_contract_stat', 'authority_broker_stat',
    'authority_bipd_file', 'authority_cargo_file', 'authority_bond_file',
    'authority_property_chk', 'authority_passenger_chk', 'authority_hhg_chk',
    'authority_private_auth_chk', 'authority_enterprise_chk',
    'authority_cargo_req', 'authority_bond_req',
    'authority_common_app_pend', 'authority_contract_app_pend', 'authority_broker_app_pend',
    'authority_common_rev_pend', 'authority_contract_rev_pend', 'authority_broker_rev_pend',
    'authority_undeliverable_mail',
    'sms_unsafe_driv_ac', 'sms_hos_driv_ac', 'sms_driv_fit_ac',
    'sms_contr_subst_ac', 'sms_veh_maint_ac',
    'co_match_confidence',
]

# Exclude SUM columns (used for LOO adjustment, not as model features)
# and sub-entity sum columns if any leaked in.
CARRIER_FEATURES = [c for c in carrier_state_agg.columns
                    if (c.startswith('carr_') or c.startswith('svc_'))
                    and not c.endswith('_sum')]

# LOO adjustment: maps feat_col -> (sum_col in carrier_state_agg, dot_col in dot_df).
# For each row where DOT is actually in the cell, we subtract its contribution.
LOO_MEAN_FEATURES = {
    'carr_mean_power':       ('carr_power_sum',       'power_units'),
    'carr_mean_crash':       ('carr_crash_sum',       'crash_count'),
    'carr_mean_insp_oos':    ('carr_insp_oos_sum',    'inspection_oos_viol'),
    'carr_mean_revoc':       ('carr_revoc_sum',       'revocation_count'),
    'carr_mean_oos':         ('carr_oos_sum',         'oos_order_count'),
    'carr_pct_hazmat':       ('carr_hazmat_sum',      'hazmat_bin'),
    'carr_pct_genfreight':   ('carr_genfreight_sum',  'crgo_genfreight'),
    'carr_pct_bldgmat':      ('carr_bldgmat_sum',     'crgo_bldgmat'),
    'carr_pct_construct':    ('carr_construct_sum',   'crgo_construct'),
    'carr_pct_livestock':    ('carr_livestock_sum',   'crgo_livestock'),
    'carr_pct_oilfield':     ('carr_oilfield_sum',    'crgo_oilfield'),
    'carr_pct_passengers':   ('carr_passengers_sum',  'crgo_passengers'),
    'carr_pct_chem':         ('carr_chem_sum',        'crgo_chem'),
    'carr_pct_meat':         ('carr_meat_sum',        'crgo_meat'),
    'carr_pct_intermodal':   ('carr_intermodal_sum',  'crgo_intermodal'),
    'carr_pct_motoveh':      ('carr_motoveh_sum',     'crgo_motoveh'),
    'carr_mean_inter_long':  ('carr_inter_long_sum',  'inter_beyond_100'),
    'carr_mean_intra_short': ('carr_intra_short_sum', 'intra_within_100'),
}

def apply_loo_adjustment(df):
    """Leave-one-out adjustment of carrier cell aggregates.
    Expects: carrier_state_agg + dot_df + dot_cell_contrib merged into df.
    Modifies the mean/pct feature columns in-place so they exclude this DOT's
    contribution to the cell they're actually placed in. For DOTs not in the
    cell (dot_n_events_in_cell=NaN -> 0), the adjustment is a no-op."""
    dot_n = df['dot_n_events_in_cell'].fillna(0).astype('float32')
    n_events = df['carr_n_events'].astype('float32')
    denom = (n_events - dot_n).clip(lower=1)
    for feat, (sum_col, dot_col) in LOO_MEAN_FEATURES.items():
        dot_val = df[dot_col].fillna(0).astype('float32')
        df[feat] = ((df[sum_col] - dot_n * dot_val) / denom).astype('float32')
    is_in_cell  = (dot_n > 0).astype('float32')
    dot_6m      = df['dot_in_6m'].fillna(0).astype('float32')
    dot_6to18m  = df['dot_in_6to18m'].fillna(0).astype('float32')
    df['carr_n_placements']      = (df['carr_n_placements']      - is_in_cell).clip(lower=0).astype('float32')
    df['carr_placements_6m']     = (df['carr_placements_6m']     - dot_6m    ).clip(lower=0).astype('float32')
    df['carr_placements_6to18m'] = (df['carr_placements_6to18m'] - dot_6to18m).clip(lower=0).astype('float32')
    df['carr_state_momentum']    = (df['carr_placements_6m']
                                     / (df['carr_placements_6to18m'] / 2.0 + 1)).astype('float32')
    return df

print(f"  DOT_NUM_FEATURES: {len(DOT_NUM_FEATURES)}")
print(f"  DOT_CAT_FEATURES: {len(DOT_CAT_FEATURES)}")
print(f"  CARRIER_FEATURES: {len(CARRIER_FEATURES)}")

# Validate feature columns exist
missing = [f for f in DOT_NUM_FEATURES + DOT_CAT_FEATURES if f not in dot_df.columns]
if missing:
    print(f"  WARNING: missing columns in dot_df: {missing}")

# Categorical encoding fit on all scorable DOTs
CAT_MAPS = {}
for c in DOT_CAT_FEATURES:
    if c not in dot_df.columns:
        continue
    vals = dot_df[c].astype('object').where(dot_df[c].notna(), 'UNK').astype(str)
    CAT_MAPS[c] = {v: i for i, v in enumerate(sorted(vals.unique()))}

def encode_dots(df):
    X_num = df[DOT_NUM_FEATURES].astype('float32').to_numpy()
    cats = []
    for c in DOT_CAT_FEATURES:
        vals = df[c].astype('object').where(df[c].notna(), 'UNK').astype(str)
        cats.append(vals.map(CAT_MAPS[c]).fillna(-1).astype('int32').to_numpy())
    X_cat = np.column_stack(cats) if cats else np.empty((len(df), 0), dtype='int32')
    return X_num, X_cat

def assemble_X(df):
    X_num, X_cat = encode_dots(df)
    X_carr = df[CARRIER_FEATURES].astype('float32').to_numpy()
    X = np.concatenate([X_num, X_carr, X_cat.astype('float32')], axis=1)
    n_num = len(DOT_NUM_FEATURES); n_carr = len(CARRIER_FEATURES)
    cat_idx = list(range(n_num + n_carr, n_num + n_carr + len(DOT_CAT_FEATURES)))
    cat_mask = np.zeros(X.shape[1], dtype=bool); cat_mask[cat_idx] = True
    return X, cat_mask, cat_idx

def make_classifier(cat_mask, cat_idx):
    if _MODEL_BACKEND == 'lightgbm':
        return lgb.LGBMClassifier(
            n_estimators=200, num_leaves=31, learning_rate=0.08,
            reg_lambda=1.0, random_state=RANDOM_SEED, verbose=-1,
            n_jobs=-1, min_child_samples=20,
        )
    return HistGradientBoostingClassifier(
        max_iter=200, max_leaf_nodes=31, learning_rate=0.08,
        l2_regularization=1.0, categorical_features=cat_mask,
        random_state=RANDOM_SEED,
    )

def fit_classifier(model, X, y, cat_idx):
    if _MODEL_BACKEND == 'lightgbm':
        X_fit = X.copy()
        if cat_idx:
            X_fit[:, cat_idx] = np.round(X_fit[:, cat_idx]).astype(np.int32)
        model.fit(X_fit, y, categorical_feature=cat_idx)
    else:
        model.fit(X, y)
    return model

def predict_scores(model, X, cat_idx):
    X_pred = X
    if _MODEL_BACKEND == 'lightgbm' and cat_idx:
        X_pred = X.copy()
        X_pred[:, cat_idx] = np.round(X_pred[:, cat_idx]).astype(np.int32)
    return model.predict_proba(X_pred)[:, 1].astype('float32')

_log("Part 3A complete")


In [ ]:
# ============================================================
# v3 Part 3B - Placement extraction + revoked pairs
# ============================================================
_log("starting Part 3B")

def extract_placements_snapshot(raw_col, line_label, extra_raw_col=None):
    cols = ['usdot', 'phy_state', 'state', raw_col] + ([extra_raw_col] if extra_raw_col else [])
    sub = dot_df[cols].copy()
    sub['dot_state'] = sub['phy_state'].fillna(sub['state'])
    def collect(row):
        out = consolidate_multi(row[raw_col])
        if extra_raw_col:
            out = list(dict.fromkeys(out + consolidate_multi(row[extra_raw_col])))
        return out
    sub['carriers'] = sub.apply(collect, axis=1)
    sub = sub[sub['carriers'].str.len() > 0]
    exploded = sub[['usdot', 'dot_state', 'carriers']].explode('carriers')
    exploded = exploded.rename(columns={'carriers': 'carrier_group', 'dot_state': 'state'})
    exploded['line'] = line_label
    # Normalise state
    exploded['state'] = exploded['state'].astype(str).str.upper().str.strip()
    return exploded[['usdot','carrier_group','state','line']].drop_duplicates().reset_index(drop=True)

def extract_placements_from_events(line_label):
    q = f"""
    WITH active AS (
        SELECT re.usdot, re.carrier_group, re.state, re.effective_date,
               ROW_NUMBER() OVER (
                   PARTITION BY re.usdot
                   ORDER BY re.effective_date DESC NULLS LAST
               ) AS rn
        FROM recent_events re
        WHERE re.line = '{line_label}'
          AND re.event_status IN ('active','active (cancellation pending)')
    )
    SELECT usdot, carrier_group, state FROM active WHERE rn = 1
    """
    p = con.execute(q).df()
    p['line'] = line_label
    p['state'] = p['state'].astype(str).str.upper().str.strip()
    return p[['usdot','carrier_group','state','line']].drop_duplicates().reset_index(drop=True)

print("Extracting placements per line...")
placements_bipd        = extract_placements_snapshot('active_bipd_raw',  'bipd')
placements_cargo       = extract_placements_snapshot('active_cargo_raw', 'cargo')
placements_wc          = extract_placements_snapshot('active_wc_raw',    'wc', extra_raw_col='wc_carrier_raw')
placements_surety_bond = extract_placements_from_events('surety_bond')
placements_surety      = extract_placements_from_events('surety')
placements_gl          = extract_placements_from_events('gl')

ALL_PLACEMENTS = {
    'bipd': placements_bipd, 'cargo': placements_cargo, 'wc': placements_wc,
    'surety_bond': placements_surety_bond, 'surety': placements_surety, 'gl': placements_gl,
}

for name, p in ALL_PLACEMENTS.items():
    kept = p.merge(carrier_state_agg[['carrier_group','state','line']],
                   on=['carrier_group','state','line'], how='inner')
    print(f"  {name:12s}: {len(p):,} raw -> {len(kept):,} in modelable cells  "
          f"({kept['usdot'].nunique():,} DOTs)")

print("\nExtracting REVOKED hard-negative pairs...")
revoked_pairs = con.execute("""
    SELECT DISTINCT usdot, carrier_group, state, line
    FROM recent_events WHERE end_reason = 'revoked'
""").df()
print(f"  {len(revoked_pairs):,} REVOKED pairs across all lines")

_log("Part 3B complete")


In [ ]:
# ============================================================
# v3 Part 3C - Training set builder (Option C) + Train 6 models with CV
# ============================================================
_log("starting Part 3C")

def build_training_optionC(placements, line_label):
    modelable = carrier_state_agg[carrier_state_agg['line'] == line_label][['carrier_group','state']]
    all_obs = placements.merge(modelable, on=['carrier_group','state'], how='inner').copy()
    all_obs = all_obs.merge(dot_df[['usdot','is_residual_market','is_cancellation_pending']],
                            on='usdot', how='left')

    clean_mask = (all_obs['is_residual_market'] == 0) & (all_obs['is_cancellation_pending'] == 0)
    pos = all_obs[clean_mask][['usdot','carrier_group','state']].copy()
    pos['y'] = np.int8(1)

    hard_rm = all_obs[all_obs['is_residual_market'] == 1][['usdot','carrier_group','state']].copy()
    hard_rm['y'] = np.int8(0)

    hard_rev = revoked_pairs[revoked_pairs['line'] == line_label][['usdot','carrier_group','state']].merge(
        modelable, on=['carrier_group','state'], how='inner'
    )
    hard_rev['y'] = np.int8(0)

    pool_by_state = {st: g['carrier_group'].to_numpy() for st, g in modelable.groupby('state')}
    k = N_NEGATIVES_PER_POSITIVE
    oversample = 2
    k_total = k * oversample

    pos_arr_u = pos['usdot'].to_numpy()
    pos_arr_s = pos['state'].to_numpy()
    pos_idx_arr = np.arange(len(pos), dtype=np.int32)
    exp_u = np.repeat(pos_arr_u, k_total)
    exp_s = np.repeat(pos_arr_s, k_total)
    exp_pos_idx = np.repeat(pos_idx_arr, k_total)
    exp_c = np.empty(len(exp_u), dtype=object)
    for st in np.unique(exp_s):
        pool = pool_by_state.get(st)
        if pool is None or len(pool) == 0: continue
        mask = exp_s == st
        n_mask = int(mask.sum())
        exp_c[mask] = pool[rng.integers(0, len(pool), size=n_mask)]

    soft = pd.DataFrame({'usdot': exp_u, 'carrier_group': exp_c, 'state': exp_s,
                         '_pos_idx': exp_pos_idx})
    soft = soft.dropna(subset=['carrier_group'])

    all_placed = pd.concat([pos[['usdot','carrier_group','state']],
                            hard_rm[['usdot','carrier_group','state']],
                            hard_rev[['usdot','carrier_group','state']]],
                           ignore_index=True).drop_duplicates()
    all_placed['_pos'] = np.int8(1)
    soft = soft.merge(all_placed, on=['usdot','carrier_group','state'], how='left')
    soft = soft[soft['_pos'].isna()].drop(columns='_pos')
    # Cap at K per positive (not per DOT) so DOTs with multiple positives
    # get proportionally more soft negatives.
    soft = soft.groupby('_pos_idx').head(k).drop(columns='_pos_idx').reset_index(drop=True)
    soft['y'] = np.int8(0)

    train = pd.concat([pos, hard_rm, hard_rev, soft], ignore_index=True)
    train['line'] = line_label
    train = train.merge(carrier_state_agg, on=['carrier_group','state','line'], how='left')
    train = train.merge(dot_df, on='usdot', how='left', suffixes=('', '_dot'))
    # LOO: when a DOT is in the cell it's labelled for, subtract its own contribution
    # from the carrier aggregates so the model can't trivially fit using the DOT's
    # own data back as a cell feature.
    train = train.merge(dot_cell_contrib, on=['usdot','carrier_group','state','line'], how='left')
    train = apply_loo_adjustment(train)
    return train, len(pos), len(hard_rm) + len(hard_rev), len(soft)

def train_fit_model(placements, line_label, n_cv_splits=3):
    print(f"\n--- Training {line_label.upper()} ---")
    _log_mem(f"[{line_label}] before build")

    train_df, n_pos, n_hard, n_soft = build_training_optionC(placements, line_label)
    if n_pos < 50:
        print(f"  insufficient clean positives ({n_pos}); skipping")
        return None

    print(f"  training rows: {len(train_df):,}  (clean_pos={n_pos:,}, hard_neg={n_hard:,}, soft_neg={n_soft:,})")
    _log_mem(f"[{line_label}] after build")

    X, cat_mask, cat_idx = assemble_X(train_df)
    y = train_df['y'].to_numpy()
    groups = train_df['usdot'].to_numpy()

    aucs, aps = [], []
    gkf = GroupKFold(n_splits=min(n_cv_splits, max(2, int(np.unique(groups).size // 100))))
    for fold, (tr, te) in enumerate(gkf.split(X, y, groups=groups)):
        m = make_classifier(cat_mask, cat_idx)
        fit_classifier(m, X[tr], y[tr], cat_idx)
        p = predict_scores(m, X[te], cat_idx)
        if np.unique(y[te]).size >= 2:
            aucs.append(roc_auc_score(y[te], p))
            aps.append(average_precision_score(y[te], p))

        if fold == 0:
            te_df = train_df.iloc[te]
            long_t  = (te_df['y'] == 1).to_numpy() & (te_df['years_with_current_carrier'] >= 3).to_numpy()
            short_t = (te_df['y'] == 1).to_numpy() & (te_df['years_with_current_carrier'] < 1).to_numpy()
            if long_t.sum() >= 20 and short_t.sum() >= 20:
                print(f"  [fold0] long-tenure positives (>=3y, n={long_t.sum()}) mean fit: {p[long_t].mean():.3f}  "
                      f"vs short-tenure (<1y, n={short_t.sum()}): {p[short_t].mean():.3f}  "
                      f"(long should be higher)")
        del m, p
        gc.collect()

    if aucs:
        print(f"  CV ROC-AUC: {np.mean(aucs):.4f} +/- {np.std(aucs):.4f} | PR-AUC: {np.mean(aps):.4f} +/- {np.std(aps):.4f}")
    else:
        print("  CV skipped (insufficient class diversity in folds)")

    final = make_classifier(cat_mask, cat_idx)
    fit_classifier(final, X, y, cat_idx)
    del X, y, groups, train_df
    gc.collect()
    _log_mem(f"[{line_label}] after refit")
    return final

print("Training fit models (6 lines)...")
MODELS = {}
for line_name, p in ALL_PLACEMENTS.items():
    MODELS[line_name] = train_fit_model(p, line_name)

_log("Part 3C complete")


In [ ]:
# ============================================================
# v3 Part 4A - Streaming per-state scoring + baselines + evaluation (6 lines)
# ============================================================
_log("starting Part 4A")

K_LONG = TOP_K_FULL_RANKING

def score_streaming(placements, model, line_label):
    if model is None or len(placements) == 0:
        return pd.DataFrame(), pd.DataFrame()

    dots = placements[['usdot', 'state']].drop_duplicates()
    dots_full = dots.merge(dot_df, on='usdot', how='left', suffixes=('', '_df'))
    dots_full = dots_full.rename(columns={'state': 'placement_state'})
    # Filter residual-market-focused carriers from the recommendation pool.
    # They remain in training (as hard negatives) so the model learns to avoid
    # them, but we don't want to recommend them to brokers.
    cells = carrier_state_agg[
        (carrier_state_agg['line'] == line_label)
        & (carrier_state_agg['carr_pct_residual_market'].fillna(0) < RESIDUAL_MARKET_CARRIER_THRESHOLD)
    ]
    observed_pairs = placements[placements['line'] == line_label][['usdot', 'carrier_group', 'state']]

    ranked_parts, obs_rank_parts = [], []
    states = sorted(cells['state'].dropna().unique())
    n_states = len(states); cands_total = 0

    for i, st in enumerate(states, 1):
        state_cells = cells[cells['state'] == st]
        state_dots = dots_full[dots_full['placement_state'] == st]
        if state_cells.empty or state_dots.empty: continue

        cross = state_dots.assign(_k=1).merge(state_cells.assign(_k=1), on='_k').drop(columns='_k')
        cross['state'] = st
        # LOO: subtract DOT's own contribution from the carrier aggregate it's placed in
        state_contrib = dot_cell_contrib[
            (dot_cell_contrib['state'] == st) & (dot_cell_contrib['line'] == line_label)
        ]
        cross = cross.merge(state_contrib, on=['usdot','carrier_group','state','line'], how='left')
        cross = apply_loo_adjustment(cross)
        cands_total += len(cross)

        X, _, cat_idx = assemble_X(cross)
        cross['fit_score'] = predict_scores(model, X, cat_idx)
        del X

        # Sort before ranking so ties on fit_score break on carrier_group deterministically.
        # DuckDB's .df() load has no stable order, so rank(method='first') alone would
        # produce different rankings across runs for tied scores.
        cross = cross.sort_values(['usdot','fit_score','carrier_group'],
                                   ascending=[True, False, True]).reset_index(drop=True)
        cross['rank']      = (cross.groupby('usdot').cumcount() + 1).astype('int32')
        cross['pool_size'] = cross.groupby('usdot')['carrier_group'].transform('size').astype('int32')

        obs_this_state = observed_pairs[observed_pairs['state'] == st]
        obs = obs_this_state.merge(
            cross[['usdot','carrier_group','rank','fit_score','pool_size']],
            on=['usdot','carrier_group'], how='inner'
        ).rename(columns={'rank': 'observed_rank'})
        obs['line'] = line_label
        obs_rank_parts.append(obs)

        top = cross.loc[cross['rank'] <= K_LONG,
                        ['usdot','carrier_group','state','fit_score','rank','pool_size']].copy()
        top['line'] = line_label
        ranked_parts.append(top)
        del cross

        if i % 10 == 0 or i == n_states:
            print(f"  [{line_label}] processed {i}/{n_states} states ({cands_total:,} candidate pairs seen)")

    ranked_long = pd.concat(ranked_parts, ignore_index=True) if ranked_parts else pd.DataFrame()
    obs_rank    = pd.concat(obs_rank_parts, ignore_index=True) if obs_rank_parts else pd.DataFrame()
    _log_mem(f"[{line_label}] after streaming scoring")
    return ranked_long, obs_rank

def eval_against_baselines(placements, obs_rank, line_label):
    if obs_rank is None or len(obs_rank) == 0:
        empty = {'n_dots':0,'top1_hit_rate':np.nan,'top5_hit_rate':np.nan,'top10_hit_rate':np.nan,'mrr':np.nan}
        return empty, empty, empty

    model_m = {
        'n_dots': len(obs_rank),
        'top1_hit_rate':  (obs_rank['observed_rank'] <= 1).mean(),
        'top5_hit_rate':  (obs_rank['observed_rank'] <= 5).mean(),
        'top10_hit_rate': (obs_rank['observed_rank'] <= 10).mean(),
        'mrr':            (1.0 / obs_rank['observed_rank']).mean(),
    }

    cells = carrier_state_agg[carrier_state_agg['line'] == line_label][['carrier_group','state']]
    pool_sizes = cells.groupby('state').size().to_dict()
    obs = placements[placements['line'] == line_label][['usdot','carrier_group','state']]

    obs_b1 = obs.merge(cells, on=['carrier_group','state'], how='inner').copy()
    obs_b1['observed_rank'] = obs_b1['state'].map(pool_sizes).apply(
        lambda n: int(rng.integers(1, max(n,1) + 1))
    )
    b1_m = {
        'n_dots': len(obs_b1),
        'top1_hit_rate':  (obs_b1['observed_rank'] <= 1).mean(),
        'top5_hit_rate':  (obs_b1['observed_rank'] <= 5).mean(),
        'top10_hit_rate': (obs_b1['observed_rank'] <= 10).mean(),
        'mrr':            (1.0 / obs_b1['observed_rank']).mean(),
    }

    all_p = placements[placements['line'] == line_label].merge(
        dot_df[['usdot','fleet_band']], on='usdot', how='left'
    )
    counts = all_p.groupby(['state','fleet_band','carrier_group']).size().reset_index(name='n')
    # Deterministic tie-break on carrier_group (see note in score_streaming).
    counts = counts.sort_values(['state','fleet_band','n','carrier_group'],
                                 ascending=[True, True, False, True]).reset_index(drop=True)
    counts['pop_rank'] = (counts.groupby(['state','fleet_band']).cumcount() + 1).astype('int32')
    pool_sz = counts.groupby(['state','fleet_band']).size().reset_index(name='pool_n')

    obs_b2 = obs.merge(cells, on=['carrier_group','state'], how='inner')
    obs_b2 = obs_b2.merge(dot_df[['usdot','fleet_band']], on='usdot', how='left')
    obs_b2 = obs_b2.merge(counts[['state','fleet_band','carrier_group','pop_rank']],
                          on=['state','fleet_band','carrier_group'], how='left')
    obs_b2 = obs_b2.merge(pool_sz, on=['state','fleet_band'], how='left')
    obs_b2['observed_rank'] = obs_b2['pop_rank'].fillna(obs_b2['pool_n']).astype('int32')
    b2_m = {
        'n_dots': len(obs_b2),
        'top1_hit_rate':  (obs_b2['observed_rank'] <= 1).mean(),
        'top5_hit_rate':  (obs_b2['observed_rank'] <= 5).mean(),
        'top10_hit_rate': (obs_b2['observed_rank'] <= 10).mean(),
        'mrr':            (1.0 / obs_b2['observed_rank'].astype('float')).mean(),
    }
    return model_m, b1_m, b2_m

print("\n" + "=" * 70)
print("SCORING (streaming per state, 6 lines)")
print("=" * 70)

SCORED = {}; OBS_RANK = {}
for line_name, placements in ALL_PLACEMENTS.items():
    ranked, obs_rank = score_streaming(placements, MODELS.get(line_name), line_name)
    SCORED[line_name] = ranked
    OBS_RANK[line_name] = obs_rank

print("\n" + "=" * 70)
print("EVALUATION - model vs baselines (per line)")
print("=" * 70)

for line_name, placements in ALL_PLACEMENTS.items():
    obs_rank = OBS_RANK[line_name]
    if MODELS.get(line_name) is None or len(obs_rank) == 0:
        print(f"\n--- {line_name.upper()} --- (skipped: no model or no scored placements)")
        continue
    print(f"\n--- {line_name.upper()} ---")
    m_m, b1_m, b2_m = eval_against_baselines(placements, obs_rank, line_name)
    summary = pd.DataFrame([
        {'approach': 'model',         **m_m},
        {'approach': 'B1 random',     **b1_m},
        {'approach': 'B2 popularity', **b2_m},
    ])
    print(summary.to_string(index=False))

if len(OBS_RANK.get('bipd', pd.DataFrame())) > 0:
    print("\n" + "=" * 70)
    print("BIPD - segmented metrics")
    print("=" * 70)
    seg = OBS_RANK['bipd'].merge(dot_df[['usdot','fleet_band']], on='usdot', how='left')

    print("\nBy state (top 10 by DOT count):")
    by_state = seg.groupby('state').agg(
        n=('usdot','count'),
        mrr=('observed_rank', lambda s: (1.0/s).mean()),
        top5=('observed_rank', lambda s: (s <= 5).mean()),
    ).sort_values('n', ascending=False).head(10)
    print(by_state.round(3).to_string())

    print("\nBy fleet band:")
    by_band = seg.groupby('fleet_band').agg(
        n=('usdot','count'),
        mrr=('observed_rank', lambda s: (1.0/s).mean()),
        top5=('observed_rank', lambda s: (s <= 5).mean()),
    ).sort_index()
    print(by_band.round(3).to_string())

_log("Part 4A complete")


In [ ]:
# ============================================================
# v3 Part 5A - Combine outputs + Option B sub-entity distance ranking
# ============================================================
_log("starting Part 5A")

non_empty_ranked = [df for df in SCORED.values() if len(df) > 0]
non_empty_obs    = [df for df in OBS_RANK.values() if len(df) > 0]

if not non_empty_ranked:
    print("No scored lines produced output. Aborting.")
    long_ranked = pd.DataFrame()
    all_obs     = pd.DataFrame()
else:
    long_ranked = pd.concat(non_empty_ranked, ignore_index=True)
    all_obs     = pd.concat(non_empty_obs,    ignore_index=True)
    print(f"Combined {len(long_ranked):,} ranked rows across lines "
          f"({sorted(long_ranked['line'].unique())})")

    observed_marker = all_obs[['usdot','line','carrier_group']].drop_duplicates()
    observed_marker['is_observed'] = np.int8(1)
    long_ranked = long_ranked.merge(observed_marker, on=['usdot','line','carrier_group'], how='left')
    long_ranked['is_observed'] = long_ranked['is_observed'].fillna(0).astype('int8')
    long_ranked = long_ranked[['usdot','line','state','rank','carrier_group',
                               'fit_score','is_observed','pool_size']]
    long_ranked = long_ranked.sort_values(['usdot','line','rank']).reset_index(drop=True)
    _log("Part 5A ranked_long combined")

    # Option B sub-entity distance ranking
    print("\nAttaching best-fit sub-entities (Option B distance)...")
    dist_feature_cols = [f'sub_{f}' for f in APPETITE_DIST_FEATURES]
    weights_arr = np.array([FEATURE_WEIGHTS.get(c, 0.0) for c in dist_feature_cols], dtype='float32')
    if weights_arr.sum() == 0:
        weights_arr = np.ones_like(weights_arr) / len(weights_arr)
    weights_arr = weights_arr / weights_arr.sum()

    sub_sum_cols = [c + '_sum' for c in dist_feature_cols]
    sub_lookup = {}
    for (g, s, l), group_df in sub_entity_centroids.groupby(['carrier_group','state','line'], sort=False):
        sub_lookup[(g, s, l)] = {
            'names': group_df['insurer_raw'].to_numpy(),
            'means': group_df[dist_feature_cols].to_numpy(dtype='float32'),
            'sums':  group_df[sub_sum_cols].to_numpy(dtype='float32'),
            'n':     group_df['sub_n_events'].to_numpy(dtype='float32'),
        }
    print(f"  built sub_lookup with {len(sub_lookup):,} (group, state, line) cells")

    # Flat dict keyed by (usdot, carrier_group, state, line, insurer_raw) -> dot_n_events.
    # Flat (one level) is lighter than nested-dict + no per-group outer entry overhead.
    dot_subent_counts = dict(zip(
        zip(dot_subent_contrib['usdot'].to_numpy(),
            dot_subent_contrib['carrier_group'].to_numpy(),
            dot_subent_contrib['state'].to_numpy(),
            dot_subent_contrib['line'].to_numpy(),
            dot_subent_contrib['insurer_raw'].to_numpy()),
        dot_subent_contrib['dot_n_events_in_subent'].to_numpy()
    ))

    def best_subentity_for_dot(dot_vals, group, state, line_label, usdot):
        entry = sub_lookup.get((group, state, line_label))
        if entry is None: return None, None
        sub_names, sub_means = entry['names'], entry['means']
        if len(sub_names) == 0: return None, None
        # LOO adjustment: subtract DOT's contribution (dot_n_events * dot_val)
        # from any sub-entity centroid where DOT actually places. Sums and ns
        # from sub_entity_centroids are over EVENTS, so a DOT with k events
        # contributes k * dot_val to the sum and k to the count.
        sums, ns = entry['sums'], entry['n']
        dot_vals_arr = np.asarray(dot_vals, dtype='float32')
        sub_vals = sub_means
        copied = False
        for i, name in enumerate(sub_names):
            dot_n = dot_subent_counts.get((usdot, group, state, line_label, name))
            if dot_n:
                if not copied:
                    sub_vals = sub_means.copy()
                    copied = True
                denom = max(float(ns[i]) - float(dot_n), 1.0)
                sub_vals[i] = (sums[i] - float(dot_n) * dot_vals_arr) / denom
        diffs = sub_vals - dot_vals
        d2 = (diffs * diffs * weights_arr).sum(axis=1)
        best_idx = int(np.argmin(d2))
        return sub_names[best_idx], float(np.sqrt(d2[best_idx]))

    top_k_per_dot_line = (long_ranked[long_ranked['rank'] <= TOP_K_PER_DOT]
                          .sort_values(['usdot','line','rank']))
    dot_dist_vals = dot_df.set_index('usdot')[APPETITE_DIST_FEATURES].astype('float32')

    best_sub_list = []
    for (usdot, line_label), group_rows in top_k_per_dot_line.groupby(['usdot','line']):
        if usdot not in dot_dist_vals.index: continue
        dot_vec = dot_dist_vals.loc[usdot].to_numpy(dtype='float32')
        for st in group_rows['state'].unique():
            state_rows = group_rows[group_rows['state'] == st]
            for grp in state_rows['carrier_group'].unique():
                sub, d = best_subentity_for_dot(dot_vec, grp, st, line_label, usdot)
                best_sub_list.append((usdot, line_label, st, grp, sub, d))

    best_sub_df = pd.DataFrame(
        best_sub_list,
        columns=['usdot','line','state','carrier_group','best_subentity','subentity_distance']
    )
    print(f"  computed {len(best_sub_df):,} best-sub-entity assignments")

    long_ranked = long_ranked.merge(
        best_sub_df[['usdot','line','state','carrier_group','best_subentity','subentity_distance']],
        on=['usdot','line','state','carrier_group'], how='left'
    )

_log("Part 5A complete")


In [ ]:
# ============================================================
# v3 Part 5B - Write artifacts (wide + long) + previews + summary
# ============================================================
_log("starting Part 5B")

TOP_K_WIDE  = TOP_K_PER_DOT
OUTPUT_WIDE = "rosella_appetite_outliers_v3.csv"
OUTPUT_LONG = "rosella_rankings_v3_long.csv"

if len(long_ranked) == 0:
    print("Nothing to write (empty long_ranked).")
else:
    long_ranked.to_csv(OUTPUT_LONG, index=False)
    print(f"Wrote {OUTPUT_LONG} ({len(long_ranked):,} rows)")

    # Build wide output: top-K recs per DOT per line, formatted as "GROUP (sub_entity)"
    top_k_slice = long_ranked[long_ranked['rank'] <= TOP_K_WIDE].copy()
    sub_mask = top_k_slice['best_subentity'].notna() & (top_k_slice['best_subentity'] != '')
    group_str = top_k_slice['carrier_group'].astype(str)
    top_k_slice['rec_str'] = np.where(
        sub_mask,
        group_str + ' (' + top_k_slice['best_subentity'].fillna('').astype(str) + ')',
        group_str,
    )
    top_k_rec = (top_k_slice.sort_values(['usdot','line','rank'])
                 .groupby(['usdot','line'])['rec_str']
                 .apply(lambda s: ' | '.join(s.values))
                 .reset_index(name=f'top{TOP_K_WIDE}_recommended'))

    # Appetite-outlier score (normalised rank)
    all_obs['appetite_outlier_score'] = (
        (all_obs['observed_rank'] - 1) / (all_obs['pool_size'] - 1).clip(lower=1)
    ).astype('float32')

    worst_line = (all_obs.sort_values(['usdot','appetite_outlier_score'], ascending=[True, False])
                  .drop_duplicates('usdot', keep='first')
                  .rename(columns={'line':'worst_line','carrier_group':'observed_carrier',
                                   'observed_rank':'observed_rank_in_line'}))
    worst_line = worst_line.merge(top_k_rec, left_on=['usdot','worst_line'],
                                  right_on=['usdot','line'], how='left').drop(columns=['line'])

    worst_line = worst_line.merge(
        dot_df[['usdot','company_name','state','phy_state','power_units','fleet_band',
                'has_authority','crash_count','inspection_oos_viol',
                'ins_distinct_insurers','ins_cancelled_count',
                'is_residual_market','is_cancellation_pending',
                'years_with_current_carrier','days_until_bipd_expiration']],
        on='usdot', how='left', suffixes=('', '_dot')
    )

    worst_line['fleet_weight'] = np.log1p(worst_line['power_units'].fillna(0)).astype('float32')
    worst_line['priority'] = (worst_line['appetite_outlier_score']
                              * (1 + 0.1 * worst_line['fleet_weight'])).astype('float32')
    worst_line = worst_line.sort_values('priority', ascending=False).reset_index(drop=True)

    cols_out = [c for c in [
        'usdot', 'company_name', 'state', 'phy_state', 'power_units', 'fleet_band',
        'worst_line', 'observed_carrier', 'observed_rank_in_line', 'pool_size',
        'appetite_outlier_score', f'top{TOP_K_WIDE}_recommended', 'priority',
        'is_residual_market', 'is_cancellation_pending', 'years_with_current_carrier',
        'days_until_bipd_expiration', 'crash_count', 'inspection_oos_viol',
        'ins_distinct_insurers', 'ins_cancelled_count', 'has_authority',
    ] if c in worst_line.columns]
    worst_line[cols_out].head(TOP_N_OUTPUT).to_csv(OUTPUT_WIDE, index=False)
    print(f"Wrote {OUTPUT_WIDE} (top {TOP_N_OUTPUT:,} DOTs; top-{TOP_K_WIDE} 'group (sub_entity)' per DOT)")

    # Previews
    print("\n" + "=" * 70)
    print("TOP 25 PREVIEW")
    print("=" * 70)
    preview_cols = [c for c in [
        'usdot','company_name','state','power_units','worst_line',
        'observed_carrier','observed_rank_in_line','pool_size',
        'appetite_outlier_score', f'top{TOP_K_WIDE}_recommended','priority'
    ] if c in worst_line.columns]
    pd.set_option('display.max_colwidth', 120)
    print(worst_line[preview_cols].head(25).to_string(index=False))

    if len(worst_line) > 0:
        print("\n" + "=" * 70)
        print("SAMPLE DOT FULL RANKING")
        print("=" * 70)
        sample_usdot = worst_line.iloc[0]['usdot']
        sample_long = (long_ranked[long_ranked['usdot'] == sample_usdot]
                       .sort_values(['line','rank']).head(30))
        print(f"USDOT {sample_usdot}: first 30 rows (across lines):")
        print(sample_long.to_string(index=False))

    print("\n" + "=" * 70)
    print("DELIVERABLE SUMMARY")
    print("=" * 70)
    print(f"Scorable DOTs ranked: {worst_line['usdot'].nunique():,}")
    print(f"Long-ranking rows written: {len(long_ranked):,}")
    print(f"\nAppetite-outlier score distribution:")
    print(worst_line['appetite_outlier_score'].describe().round(3).to_string())
    print(f"\nPriority distribution:")
    print(worst_line['priority'].describe().round(3).to_string())
    print(f"\n[Total runtime: {_elapsed()}]")

_log("Part 5B complete")
